# 👁️ VISION LIVE v5 — une IA qui voit, vérifie, apprend de tes corrections, se souvient, répond à ta voix, SURVEILLE et COMPREND

Tout tourne **en local** sur ta carte graphique. Seules les alertes Telegram passent par internet (si tu configures un bot).
Les modèles sont dans le dossier commun `~/Modeles_IA/` (partagé avec tes autres projets, voir `LISEZMOI.md`).

| Brique | Modèle | Rôle |
|---|---|---|
| 👁️ Vision | **YOLOE-11m** (medium, vocabulaire ouvert) | repère vite ~230 objets décrits en texte + tous ceux que tu lui apprends |
| 🔎 2e regard | **MobileCLIP2-S2** (Apple) | regarde chaque objet **de près** (seulement ses pixels) : confirme, corrige ou rejette (mains, flou, peau...) |
| 🧠 Suivi | BoT-SORT + vote sur 15 images + hystérésis | un objet garde son identité et son nom ; il n'apparaît qu'après 3 images |
| 🤔 Honnêteté | catégories | quand il hésite, il le dit : *« aliment (pain ?) »* en couleur éteinte |
| ✍️ **Niveau 2** | mémoire visuelle de tes corrections | tu corriges une fois → il suit ta correction **immédiatement** (« (appris) ») |
| 🏋️ **Niveau 3** | adaptateur entraîné + signatures YOLOE adaptées | il s'entraîne sur **tes** exemples (≈ 0,2 s), tout seul toutes les 10 corrections |
| 🎓 Objets perso | prompts visuels YOLOE | tu entoures **ton** objet → il le reconnaît ensuite (en or ★) |
| 📍 Mémoire | fichier JSON + photos | « où est mon téléphone ? » → *« vu il y a 4 min, à gauche, près de : ordinateur »* |
| 🎙️ Voix | **Whisper small** (français) + synthèse vocale | tu parles, il comprend et il répond à voix haute |
| 🚨 **Surveillance** | règles + **YuNet** (visages) + H.264 | zones, alertes intrus / animal / objet disparu, clip 10 s avant + 10 s après, meilleur visage, Telegram |
| 🧠 **Niveau 4** | **Qwen3-VL-4B** (4 bits, à la demande) | **comprend** la scène et répond en français ; décrit chaque alerte |

**Précision mesurée** sur 74 vraies détections de ta session du 29/10 (`bench/test_vignettes.py`, validation croisée) :
YOLOE seul **19 %** → + vérificateur **59 %** → + apprentissage **80 %** de noms justes ; 19 fausses détections sur 24 écartées.
**Surveillance** (`bench/test_surveillance.py`, vidéo de référence) : alerte en 1,6 s, visage trouvé, + 0,07 ms par image.
**Qwen** (`bench/test_memoire_qwen.py`) : réveil ~6 s, ~6-7 s par réponse ; tout tient en mémoire (pic 5,6 Go sur 5,7 utilisables) ;
pendant qu'il réfléchit, la fenêtre passe d'environ 21 à 11 images/s.

### 🚨 Surveillance
1. (facultatif) `Z` → clique les coins d'une zone (bureau, porte...) → `Entrée` → tape son nom. Sans zone, toute l'image est surveillée.
2. `G` → **armée dans 10 s** (le temps de sortir). `G` à nouveau = désarmée. Qwen se réveille dès qu'on arme.
3. Règles par défaut (cellule 7c, modifiables) : **personne** présente 1,5 s · **animal** présent 2 s · **téléphone / ordinateur / portefeuille / tablette / clé** posé 5 s puis disparu 5 s.
4. Chaque alerte → `evenements/AAAAMMJJ_HHMMSS_regle/` : `image.jpg`, `clip.mp4`, `visage.jpg`, `description.txt` (Qwen), `info.json`.
5. **Telegram** : crée un bot avec @BotFather, mets son jeton dans `config/telegram.json` → `{"token": "123456:ABC..."}`,
   puis envoie un message à ton bot (ce chat devient le seul autorisé). Commandes : `/photo` `/scene` `/armer` `/desarmer` `/etat` `/events`.
   Sans bot ou sans internet, les alertes attendent dans `evenements/alertes_en_attente.jsonl` et partent plus tard.

### Ce que tu peux lui dire (touche `V`, puis parle) ou écrire (touche `F`)
- « **Où est** mon téléphone ? » · « Tu as vu mes lunettes ? »
- « **Trouve** ma clé USB » · « Cherche une bouteille » → la cible est verrouillée à l'écran
- « **Qu'est-ce que tu vois ?** » (liste rapide) · « **Combien** de bouteilles ? »
- 🧠 « **Que fait** la personne ? » · « **Qu'est-ce qui se passe** ? » · « **De quelle couleur** est sa chemise ? » · « **Explique**... » → **Qwen**
  (toute question que le système ne sait pas traiter seul part aussi chez Qwen)
- « **Non, c'est un cordon badge** » · « Ce n'est pas un objet » → corrige l'objet sous le viseur (au centre)
- « Passe en **mode nuit** » · « **Haute précision** » · « Lance une **mission** » · « **Stop** »

### Touches
| Touche | Action |
|---|---|
| `I` | 🧠 **Qwen** : « que se passe-t-il ? » sur l'image en direct (réponse écrite + à voix haute) |
| `G` | **Surveillance** : armer (dans 10 s) / désarmer |
| `Z` | **Zone** : clique les coins, `Entrée`, tape le nom (`Retour` = effacer le dernier coin, ou la dernière zone) |
| `V` | Parler (appuie, parle, ré-appuie — ou attends 6 s) |
| `F` | Écrire une question |
| `C` | **Corriger** : clique sur l'objet → `1` `2` `3` = ses propositions, `X` = pas un objet, ou tape le bon nom + `Entrée` |
| `E` | **Réviser** les objets dont il n'était pas sûr (mêmes touches, `Espace` = passer) |
| `U` | **Entraîner** maintenant sur tes corrections (il le fait aussi tout seul toutes les 10) |
| `A` | **Apprendre un nouvel objet** perso : entoure-le, tape son nom, `Entrée` |
| `1` `2` `3` `4` | HUD · Spotlight · Heatmap · Vision nocturne |
| `H` | Haute précision (petits objets) |
| `M` | Mission (mini-jeu) |
| `+` / `-` | Seuil de confiance |
| `T` / `S` / `R` | Traînées / Capture / Enregistrement vidéo |
| `Q` | Quitter |

💡 **Plus tu corriges, plus il devient précis** : 5 minutes de `E` (révision) par jour suffisent.

Fichiers : `~/Documents/Vision_Live/` — `evenements/` (alertes), `dataset/` (tes corrections, format YOLO), `bench/` (tests),
`memoire/`, `captures/`, `config/` (zones, Telegram), `logs/` ; modèles : `~/Modeles_IA/`.

In [ ]:
# ⚡ LANCEMENT RAPIDE — exécute SEULEMENT cette cellule : elle charge tout le projet et ouvre la vision.
# (Elle relit les cellules ci-dessous depuis le fichier : pense à sauvegarder le notebook (Ctrl+S) après une modification.)
%run ~/Documents/Vision_Live/lancer_vision.py

In [ ]:
# 1) Installation (une seule fois) — torch/torchvision/transformers déjà présents, ils ne seront pas modifiés
%pip install -q ultralytics lap

import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# 2) Configuration du projet
import os
os.environ.setdefault("QT_QPA_PLATFORM", "xcb")   # fenêtre OpenCV fiable sous Wayland (via XWayland)
os.environ.setdefault("HF_HUB_OFFLINE", "1")      # tous les modèles sont sur le disque : pas besoin d'internet
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

from pathlib import Path
import time, csv, random, threading, json, re, queue, subprocess, unicodedata, difflib, contextlib
from collections import defaultdict, deque, Counter
from datetime import datetime

import cv2
import numpy as np
import torch

PROJET   = Path.home() / "Documents" / "Vision_Live"
MODELES  = Path.home() / "Modeles_IA"    # dossier COMMUN à tous tes projets (YOLOE, visages, Qwen...) — voir LISEZMOI.md
CAPTURES = PROJET / "captures"
LOGS     = PROJET / "logs"
MEMOIRE  = PROJET / "memoire"            # objets appris + où chaque objet a été vu
VIGNETTES = MEMOIRE / "vignettes"
REGLAGES = PROJET / "config"
for d in (MODELES, CAPTURES, LOGS, MEMOIRE, VIGNETTES, REGLAGES):
    d.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "camera": 0,                  # /dev/video0
    "largeur": 1280,
    "hauteur": 720,
    "modele": "yoloe-11m-seg.pt", # YOLOE medium : vocabulaire ouvert (on lui donne les objets en texte ou en image)
    "imgsz": 640,                 # précision normale
    "imgsz_hd": 960,              # touche H : haute précision (petits objets : clé USB, stylo, bague...)
    "conf": 0.30,                 # seuil d'AFFICHAGE (confiance lissée d'un objet suivi)
    "conf_suivi": 0.10,           # seuil donné au tracker : il garde les objets même quand le score baisse un instant
    "age_min": 3,                 # un objet n'apparaît qu'après 3 images d'affilée (fini les fausses alertes d'une image)
    "vote": 15,                   # nb d'images pour stabiliser le nom d'un objet suivi
    # --- 2e regard (MobileCLIP2) — réglages mesurés sur 74 vraies vignettes (bench/), précision 19 % → 59 %
    "verif": True,
    "verif_modele": "hf-hub:timm/MobileCLIP2-S2-OpenCLIP",
    "verif_rejet": 0.40,          # « main / flou / mur / peau » à plus de 40 %...
    "verif_top_rejet": 0.40,      # ...et aucun objet à plus de 40 % → objet ignoré
    "verif_perdu": 0.20,          # le vérificateur ne reconnaît aucun objet à plus de 20 % → ignoré (presque toujours faux)
    "verif_alpha": 0.50,          # poids de YOLOE pour départager les propositions du vérificateur
    "verif_doute": 0.60,          # si le 2e nom vaut plus de 60 % du 1er → « catégorie (nom ?) »
    # --- apprentissage par tes corrections (niveaux 2 et 3) — mesuré : 59 % → 80 % de précision
    "memoire_seuil": 0.88,        # ressemblance à partir de laquelle il suit directement ta correction
    "memoire_k": 3,               # nb d'exemples proches consultés
    "entrainement_min": 8,        # nb minimum d'exemples pour entraîner l'adaptateur
    "entrainement_auto": 10,      # ré-entraîne tout seul toutes les 10 corrections (et touche U)
    "entrainement_epoques": 60,
    "entrainement_reg": 0.10,     # garde l'adaptateur proche du modèle d'origine (évite d'« oublier » le reste)
    "proto_min": 3,               # objet corrigé au moins 3 fois → sa signature dans YOLOE est adaptée à tes images
    "proto_poids": 0.50,          # part de tes images dans cette signature ; mesuré : 0.3 = aucun effet,
                                  # 0.5 = la classe corrigée passe devant (0.63 contre 0.37), 0.7 = trop (oublie le texte)
    "revoir_max": 100,            # objets douteux mis de côté pour la révision (touche E)
    "device": 0 if torch.cuda.is_available() else "cpu",
    "quantize": 16 if torch.cuda.is_available() else None,   # FP16 sur GPU
    "panneau": 400,               # largeur du panneau latéral (px)
    "whisper": "openai/whisper-small",   # reconnaissance vocale locale (français)
    "voix_max_s": 6,              # durée max d'une question vocale
}

# Réglages du tracker (BoT-SORT) : garde un objet 2 s s'il disparaît, et accepte les scores bas d'un objet déjà suivi.
# (ReID désactivé : le mode "auto" d'ultralytics n'est pas compatible avec YOLOE.)
TRACKER = REGLAGES / "tracker.yaml"
TRACKER.write_text("""tracker_type: botsort
track_high_thresh: 0.25
track_low_thresh: 0.10
new_track_thresh: 0.35
track_buffer: 60
match_thresh: 0.80
fuse_score: True
gmc_method: none
proximity_thresh: 0.5
appearance_thresh: 0.8
with_reid: False
model: auto
""")

GPU_NOM = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

@contextlib.contextmanager
def dans_dossier(d):
    """Exécute un bloc dans un autre dossier (ultralytics cherche certains fichiers dans le dossier courant)."""
    ancien = os.getcwd(); os.chdir(d)
    try: yield
    finally: os.chdir(ancien)

def sans_accents(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()

print("Dossier projet :", PROJET)
print("Modèles        :", MODELES)
print("Calcul sur     :", GPU_NOM)

In [ ]:
# 3) Vocabulaire : ce que le modèle sait reconnaître (anglais → français), rangé par catégorie
# YOLOE comprend du TEXTE : chaque ligne devient un objet détectable. Tu peux en ajouter autant que tu veux.
# Pas de doublons (ex. "phone" + "smartphone") ni de choses trop vagues ("main", "mur") :
# ils se volent les détections entre eux et font baisser la précision.
# Les catégories servent quand le système hésite entre deux voisins : il affiche "aliment (pain ?)".
CATEGORIES = {
    "personne": {"person": "personne"},
    "accessoire": {
        "baseball cap": "casquette", "hat": "chapeau", "eyeglasses": "lunettes", "sunglasses": "lunettes de soleil",
        "face mask": "masque", "wristwatch": "montre", "ring": "bague", "necklace": "collier", "earphones": "écouteurs",
        "headphones": "casque audio", "tie": "cravate", "belt": "ceinture", "lanyard": "cordon badge", "badge": "badge",
        "id card": "carte d'identité"},
    "vêtement": {
        "polo shirt": "polo", "t-shirt": "t-shirt", "jacket": "veste", "dress": "robe", "shoe": "chaussure", "sandal": "sandale"},
    "électronique": {
        "cell phone": "téléphone", "usb flash drive": "clé USB", "laptop": "ordinateur portable", "computer monitor": "écran",
        "television": "télévision", "computer keyboard": "clavier", "computer mouse": "souris", "tablet computer": "tablette",
        "charger": "chargeur", "power bank": "batterie externe", "cable": "câble", "power strip": "multiprise",
        "remote control": "télécommande", "camera": "appareil photo", "speaker": "enceinte", "microphone": "micro",
        "calculator": "calculatrice", "game controller": "manette", "hard drive": "disque dur", "router": "routeur",
        "printer": "imprimante", "light bulb": "ampoule", "flashlight": "lampe torche", "battery": "pile",
        "electric fan": "ventilateur", "desk lamp": "lampe"},
    "bureau": {
        "pen": "stylo", "pencil": "crayon", "marker": "marqueur", "book": "livre", "notebook": "cahier", "paper": "feuille",
        "envelope": "enveloppe", "folder": "classeur", "scissors": "ciseaux", "stapler": "agrafeuse", "tape": "scotch",
        "ruler": "règle", "eraser": "gomme", "sticky note": "post-it", "calendar": "calendrier",
        "business card": "carte de visite", "receipt": "ticket"},
    "argent & clés": {
        "credit card": "carte bancaire", "wallet": "portefeuille", "banknote": "billet", "coin": "pièce", "key": "clé",
        "keychain": "porte-clés", "padlock": "cadenas"},
    "sac": {
        "backpack": "sac à dos", "handbag": "sac à main", "plastic bag": "sac plastique", "suitcase": "valise",
        "box": "boîte", "cardboard box": "carton", "basket": "panier"},
    "cuisine": {
        "bottle": "bouteille", "water bottle": "gourde", "can": "canette", "cup": "tasse", "mug": "mug", "glass": "verre",
        "plastic cup": "gobelet", "wine glass": "verre à vin", "plate": "assiette", "bowl": "bol", "fork": "fourchette",
        "knife": "couteau", "spoon": "cuillère", "pot": "marmite", "pan": "poêle", "kettle": "bouilloire",
        "thermos": "thermos", "lighter": "briquet", "matches": "allumettes", "napkin": "serviette", "tissue": "mouchoir",
        "straw": "paille", "food container": "boîte repas", "jar": "bocal"},
    "aliment": {
        "banana": "banane", "apple": "pomme", "orange": "orange", "mango": "mangue", "pineapple": "ananas",
        "avocado": "avocat", "coconut": "noix de coco", "lemon": "citron", "tomato": "tomate", "onion": "oignon",
        "chili pepper": "piment", "carrot": "carotte", "potato": "pomme de terre", "cassava": "manioc", "plantain": "plantain",
        "bread": "pain", "sandwich": "sandwich", "cake": "gâteau", "cookie": "biscuit", "candy": "bonbon",
        "chocolate": "chocolat", "egg": "œuf", "fish": "poisson", "meat": "viande", "pizza": "pizza"},
    "meuble": {
        "chair": "chaise", "plastic chair": "chaise en plastique", "table": "table", "desk": "bureau", "sofa": "canapé",
        "bed": "lit", "shelf": "étagère", "cabinet": "armoire", "bench": "banc"},
    "maison": {
        "pillow": "oreiller", "blanket": "couverture", "curtain": "rideau", "door": "porte", "window": "fenêtre",
        "mirror": "miroir", "clock": "horloge", "picture frame": "cadre photo", "vase": "vase", "potted plant": "plante en pot",
        "candle": "bougie", "trash can": "poubelle", "bucket": "seau", "broom": "balai", "umbrella": "parapluie",
        "parasol": "parasol", "tent": "tente", "hand fan": "éventail", "air conditioner": "climatiseur",
        "refrigerator": "frigo", "microwave": "micro-ondes", "stove": "cuisinière", "sink": "évier",
        "washing machine": "machine à laver", "toilet": "toilettes", "hanger": "cintre"},
    "toilette & santé": {
        "towel": "serviette de bain", "soap": "savon", "toothbrush": "brosse à dents", "toothpaste": "dentifrice",
        "comb": "peigne", "hairbrush": "brosse à cheveux", "hair dryer": "sèche-cheveux", "perfume": "parfum",
        "lotion": "crème", "lipstick": "rouge à lèvres", "nail polish": "vernis", "medicine": "médicament"},
    "outil": {
        "rope": "corde", "extension cord": "rallonge", "hammer": "marteau", "screwdriver": "tournevis",
        "wrench": "clé à molette", "pliers": "pince", "drill": "perceuse", "nail": "clou", "glue": "colle",
        "paint brush": "pinceau", "gloves": "gants", "helmet": "casque"},
    "loisir": {
        "ball": "ballon", "teddy bear": "nounours", "toy": "jouet", "doll": "poupée", "playing cards": "cartes à jouer",
        "dice": "dé", "guitar": "guitare", "drum": "tambour", "balloon": "ballon de baudruche"},
    "extérieur": {
        "car": "voiture", "motorcycle": "moto", "bicycle": "vélo", "bus": "bus", "truck": "camion", "tire": "pneu",
        "wheelbarrow": "brouette", "palm tree": "palmier", "tree": "arbre", "flower": "fleur", "rock": "caillou",
        "house": "maison", "fence": "clôture", "gate": "portail", "flag": "drapeau", "sign": "panneau"},
    "animal": {
        "dog": "chien", "cat": "chat", "bird": "oiseau", "chicken": "poule", "goat": "chèvre", "cow": "vache",
        "sheep": "mouton", "pig": "cochon", "horse": "cheval", "lizard": "lézard", "insect": "insecte",
        "butterfly": "papillon", "mosquito": "moustique", "ant": "fourmi", "spider": "araignée"},
}
VOCAB = {en: fr for groupe in CATEGORIES.values() for en, fr in groupe.items()}          # anglais → français
CATEGORIE_DE = {fr: cat for cat, groupe in CATEGORIES.items() for fr in groupe.values()}  # français → catégorie
print(f"Vocabulaire : {len(VOCAB)} objets en {len(CATEGORIES)} catégories")

In [ ]:
# 4) Le CERVEAU : YOLOE + vocabulaire texte + objets appris par l'image + recherches libres
from ultralytics import YOLOE
from ultralytics.models.yolo.yoloe import YOLOEVPSegPredictor

class Cerveau:
    """Tient la liste de tout ce que le modèle cherche, et la recharge dans YOLOE quand elle change.
    - vocabulaire : objets décrits en texte (cellule 3)
    - perso       : TES objets, appris en les entourant à la souris (touche A)
    - recherches  : objets demandés en direct ("trouve une bouteille bleue")
    - prototypes  : niveau 3 — pour les objets que tu as corrigés plusieurs fois, la signature texte est
                    mélangée avec la signature de TES images (cellule 4c)"""
    FICHIER_PERSO = MEMOIRE / "objets_perso.npz"
    CACHE_TEXTE = MODELES / "cache_textes.pt"
    # Apparence réelle des objets, apprise sur Open Images (bench/yoloe_open_images.py) : mélangée au texte.
    # Mesuré sur 571 objets jamais vus : objets retrouvés 40 % → 59 %, noms justes 57 % → 62 % (poids 0,5 ; 0,3 et 0,7 moins bien)
    FICHIER_PUBLICS = PROJET / "dataset" / "open_images" / "prototypes_yoloe.pt"

    def __init__(self):
        self.model = YOLOE(str(MODELES / CONFIG["modele"]))
        self.model.to("cuda" if CONFIG["device"] == 0 else "cpu")
        self.cache = torch.load(self.CACHE_TEXTE) if self.CACHE_TEXTE.exists() else {}   # texte -> embedding
        self.perso = {}          # nom -> embedding moyen (512,)
        self.perso_n = Counter() # nom -> nb d'exemples appris
        self.recherches = {}     # texte anglais/libre -> nom affiché
        self.prototypes = {}     # nom -> signature moyenne de tes images (niveau 3)
        self.publics = torch.load(self.FICHIER_PUBLICS) if self.FICHIER_PUBLICS.exists() else {}   # nom -> Open Images
        self._charger_perso()
        self.appliquer()

    # ---------- embeddings
    def _textes(self, textes):
        manquants = [t for t in textes if t not in self.cache]
        if manquants:
            with dans_dossier(MODELES):   # l'encodeur de texte MobileCLIP est rangé dans models/
                pe = self.model.model.get_text_pe(manquants, cache_clip_model=True)[0].float().cpu()
            self.cache.update(zip(manquants, pe))
            torch.save(self.cache, self.CACHE_TEXTE)
        return torch.stack([self.cache[t] for t in textes])

    def embedding_visuel(self, frame, boite):
        """Transforme 'cette zone de l'image' en signature que YOLOE sait rechercher."""
        p = YOLOEVPSegPredictor(overrides=dict(task="segment", mode="predict", save=False, batch=1, verbose=False,
                                               imgsz=CONFIG["imgsz"], device=CONFIG["device"]))
        p.set_prompts({"bboxes": np.array([boite], dtype=np.float32), "cls": np.array([0])})
        p.setup_model(model=self.model.model, verbose=False)
        vpe = p.get_vpe(frame)[0, 0].float().cpu()
        self.model.predictor = None
        return torch.nn.functional.normalize(vpe, dim=-1)

    # ---------- liste des classes
    def appliquer(self):
        """Recharge toutes les classes dans le modèle (le tracker repart à zéro)."""
        textes = list(VOCAB) + [t for t in self.recherches if t not in VOCAB]
        self.noms_fr = [VOCAB.get(t) or self.recherches[t] for t in textes] + list(self.perso)
        self.textes_en = textes + [None] * len(self.perso)      # utilisé par le vérificateur (None = objet perso)
        self.est_perso = [False] * len(textes) + [True] * len(self.perso)
        pe = self._textes(textes)
        bp = CONFIG.get("proto_public_poids", 0.5)
        for i, nom in enumerate(self.noms_fr[:len(textes)]):    # apparence réelle (Open Images)
            if nom in self.publics and bp:
                pe[i] = torch.nn.functional.normalize((1 - bp) * pe[i] + bp * self.publics[nom], dim=-1)
        for i, nom in enumerate(self.noms_fr[:len(textes)]):    # niveau 3 : signature adaptée à TES images
            if nom in self.prototypes:
                b = CONFIG["proto_poids"]
                pe[i] = torch.nn.functional.normalize((1 - b) * pe[i] + b * self.prototypes[nom], dim=-1)
        if self.perso:
            pe = torch.cat([pe, torch.stack(list(self.perso.values()))])
        device = next(self.model.model.parameters()).device
        self.model.set_classes([f"c{i}" for i in range(len(pe))], pe[None].to(device))
        self.generation = getattr(self, "generation", 0) + 1

    def apprendre(self, nom, frame, boite):
        """Ajoute un exemple visuel. Plusieurs exemples du même objet → signature moyenne plus robuste."""
        e = self.embedding_visuel(frame, boite)
        n = self.perso_n[nom]
        moy = e if n == 0 else torch.nn.functional.normalize(self.perso[nom] * n + e, dim=-1)
        self.perso[nom], self.perso_n[nom] = moy, n + 1
        cv2.imwrite(str(VIGNETTES / f"perso_{sans_accents(nom).replace(' ', '_')}.jpg"),
                    frame[boite[1]:boite[3], boite[0]:boite[2]])
        self._sauver_perso()
        self.appliquer()
        return n + 1

    def oublier(self, nom):
        if nom in self.perso:
            del self.perso[nom], self.perso_n[nom]
            self._sauver_perso(); self.appliquer()

    def chercher_texte(self, texte_en, nom_fr):
        """Ajoute un objet libre au vocabulaire (ex. 'blue bottle')."""
        if texte_en not in VOCAB and texte_en not in self.recherches:
            self.recherches[texte_en] = nom_fr
            self.appliquer()

    def _sauver_perso(self):
        noms = list(self.perso)
        np.savez(self.FICHIER_PERSO, noms=np.array(noms, dtype=np.str_),
                 pe=np.stack([self.perso[n].numpy() for n in noms]) if noms else np.zeros((0, 512), np.float32),
                 n=np.array([self.perso_n[n] for n in noms]))

    def _charger_perso(self):
        if self.FICHIER_PERSO.exists():
            d = np.load(self.FICHIER_PERSO)
            for nom, pe, n in zip(d["noms"].tolist(), d["pe"], d["n"]):
                self.perso[nom], self.perso_n[nom] = torch.from_numpy(pe), int(n)

    def nom(self, c):
        return self.noms_fr[c]

cerveau = Cerveau()
model = cerveau.model

# Préchauffage + mesure de vitesse
_dummy = np.zeros((CONFIG["hauteur"], CONFIG["largeur"], 3), dtype=np.uint8)
for sz in (CONFIG["imgsz"], CONFIG["imgsz_hd"]):
    for _ in range(3):
        model.predict(_dummy, imgsz=sz, quantize=CONFIG["quantize"], device=CONFIG["device"], verbose=False)
    t0 = time.perf_counter()
    for _ in range(10):
        model.predict(_dummy, imgsz=sz, quantize=CONFIG["quantize"], device=CONFIG["device"], verbose=False)
    ms = (time.perf_counter() - t0) / 10 * 1000
    print(f"imgsz {sz}: ~{ms:.1f} ms/image (~{1000/ms:.0f} FPS max)")
print(f"Cerveau prêt : {len(cerveau.noms_fr)} objets connus dont {len(cerveau.perso)} appris par toi, "
      f"{len(cerveau.publics)} adaptés à leur apparence réelle (Open Images)")
print(f"VRAM utilisée : {torch.cuda.memory_allocated()/1e9:.2f} Go")

In [ ]:
# 4b) Le VÉRIFICATEUR : un 2e regard (MobileCLIP2 d'Apple) sur chaque objet, découpé en pleine résolution
# YOLOE repère vite les zones sur l'image réduite ; le vérificateur regarde chaque objet de près (seulement ses
# pixels, grâce au masque de segmentation) et confirme, corrige, ou rejette. Il tourne en arrière-plan.
# Il utilise aussi ce que TU lui as appris (cellule 4c) : mémoire visuelle de tes corrections + adaptateur entraîné.
import open_clip
from timm.utils.model import reparameterize_model
from PIL import Image

PHRASES = ["a photo of a {}.", "a close-up photo of a {}.", "a photo of a {} held in a hand.", "a low-light photo of a {}."]
# « Non-objets » : ce que YOLOE prend le plus souvent pour un objet (mains, peau, flou...).
# Pas de "fenêtre lumineuse" ni de "tissu" : ils voleraient les vraies fenêtres et les vêtements.
REJETS = ["a blurry photo", "a plain painted wall", "a dark shadow", "an empty background", "a smudge of noise",
          "a human hand", "a human face", "a patch of human skin", "a human arm", "a human eye"]

class Verificateur:
    CACHE = MODELES / "cache_verif.pt"

    def __init__(self):
        gpu = torch.cuda.is_available()
        self.dev, self.dtype = ("cuda", torch.float16) if gpu else ("cpu", torch.float32)
        m, _, self.pre = open_clip.create_model_and_transforms(CONFIG["verif_modele"])
        self.model = reparameterize_model(m.eval()).to(self.dev, self.dtype)
        self.tok = open_clip.get_tokenizer("ViT-B-32")    # tokenizer CLIP standard (celui du dépôt remplit mal les phrases)
        self.echelle = float(self.model.logit_scale.detach().exp())
        self.cache = torch.load(self.CACHE) if self.CACHE.exists() else {}
        self.R = self._phrases(REJETS, gabarits=["{}"])
        self.textes, self.gen = None, None
        self.apprentissage = None                         # branché par la cellule 4c
        self.jobs, self.res = queue.Queue(maxsize=64), {}
        threading.Thread(target=self._boucle, daemon=True).start()

    @torch.no_grad()
    def _phrases(self, noms, gabarits=PHRASES):
        """Signature texte de chaque nom = moyenne de plusieurs phrases (plus robuste qu'une seule)."""
        manquants = [n for n in noms if (n, len(gabarits)) not in self.cache]
        for i in range(0, len(manquants), 64):
            lot = manquants[i:i + 64]
            t = self.tok([g.format(n) for n in lot for g in gabarits]).to(self.dev)
            e = torch.nn.functional.normalize(self.model.encode_text(t).float(), dim=-1)
            e = torch.nn.functional.normalize(e.view(len(lot), len(gabarits), -1).mean(1), dim=-1).cpu()
            self.cache.update({(n, len(gabarits)): v for n, v in zip(lot, e)})
        if manquants:
            torch.save(self.cache, self.CACHE)
        return torch.stack([self.cache[(n, len(gabarits))] for n in noms]).to(self.dev, self.dtype)

    def preparer(self, cerveau):
        """À appeler quand la liste des classes change (nouvel objet appris, recherche libre...)."""
        T = self._phrases([t or "object" for t in cerveau.textes_en])
        self.textes, self.gen = torch.cat([T, self.R]), cerveau.generation
        self.n_classes = len(T)

    @staticmethod
    def decouper(frame, boite, parts=None):
        """Découpe l'objet ; si on a son masque, tout ce qui n'est pas l'objet devient gris
        (sinon un cordon sur un polo est « noyé » dans le polo)."""
        x1, y1, x2, y2 = boite
        m = int(0.08 * max(x2 - x1, y2 - y1))
        X1, Y1, X2, Y2 = max(0, x1 - m), max(0, y1 - m), min(frame.shape[1], x2 + m), min(frame.shape[0], y2 + m)
        crop = frame[Y1:Y2, X1:X2].copy()
        if parts:
            masque = np.zeros(crop.shape[:2], np.uint8)
            cv2.fillPoly(masque, [p - [X1, Y1] for p in parts], 255)
            if masque.mean() > 255 * 0.08:                   # masque exploitable (pas un pixel perdu)
                masque = cv2.dilate(masque, np.ones((9, 9), np.uint8))
                crop[masque == 0] = 127
        return cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

    def demander(self, uid, frame, boite, parts=None):
        x1, y1, x2, y2 = boite
        if min(x2 - x1, y2 - y1) < 20:
            return
        try:
            self.jobs.put_nowait((uid, self.decouper(frame, boite, parts), self.gen))
        except queue.Full:
            pass

    @torch.no_grad()
    def encoder(self, crops_rgb):
        """Signature image (512 nombres) de chaque découpe, AVANT adaptation."""
        X = torch.stack([self.pre(Image.fromarray(c)) for c in crops_rgb]).to(self.dev, self.dtype)
        return torch.nn.functional.normalize(self.model.encode_image(X).float(), dim=-1)

    @torch.no_grad()
    def probas(self, I):
        """Probabilités sur [classes..., rejets...] ; passe d'abord par l'adaptateur entraîné sur TA pièce s'il existe."""
        a = self.apprentissage
        if a is not None and a.W is not None:
            I = torch.nn.functional.normalize(I + I @ a.W, dim=-1)
        return (self.echelle * I @ self.textes.float().T).softmax(-1).cpu().numpy()

    def classer(self, crops_rgb):
        """Probabilités pour une liste d'images (utilisé aussi par les tests du banc d'essai)."""
        return self.probas(self.encoder(crops_rgb))

    def _boucle(self):
        while True:
            lot = [self.jobs.get()]
            while not self.jobs.empty() and len(lot) < 16:
                lot.append(self.jobs.get_nowait())
            lot = [j for j in lot if j[2] == self.gen]          # ignore les demandes d'une ancienne liste de classes
            if not lot or self.textes is None:
                continue
            try:
                I = self.encoder([c for _, c, _ in lot])
                P = self.probas(I)
                souvenirs = self.apprentissage.souvenir(I) if self.apprentissage else [None] * len(lot)
            except Exception as e:
                print("vérificateur :", e); continue
            for (uid, _, gen), p, mem in zip(lot, P, souvenirs):
                ancien = self.res.get(uid)
                if ancien and ancien["gen"] == gen:              # moyenne des vérifications successives
                    k = ancien["n"]
                    p = (ancien["p"] * k + p) / (k + 1)
                    mem = mem or ancien.get("memoire")
                else:
                    k = 0
                self.res[uid] = {"p": p, "n": k + 1, "gen": gen, "memoire": mem}

verificateur = Verificateur() if CONFIG["verif"] else None
if verificateur:
    t0 = time.time(); verificateur.preparer(cerveau)
    verificateur.classer([np.zeros((64, 64, 3), np.uint8)] * 16)
    t1 = time.time(); verificateur.classer([np.zeros((64, 64, 3), np.uint8)] * 16)
    print(f"Vérificateur prêt : {verificateur.n_classes} classes + {len(REJETS)} rejets | "
          f"16 objets vérifiés en {(time.time() - t1) * 1000:.0f} ms | préparation {t1 - t0:.1f} s")
    print(f"VRAM utilisée : {torch.cuda.memory_allocated()/1e9:.2f} Go")

In [ ]:
# 4c) APPRENTISSAGE — niveaux 2 et 3 : il apprend de TES corrections
# Niveau 2 : chaque correction est gardée (image + signature). Dès qu'un objet ressemble fort à un exemple corrigé,
#            le système suit ta correction : effet IMMÉDIAT, sans entraînement.
# Niveau 3 : à partir de tes exemples, il ENTRAÎNE un adaptateur (petite couche du vérificateur) et ajuste les
#            signatures du détecteur YOLOE pour TES objets. Mesuré (validation croisée, 74 vignettes réelles,
#            bench/test_vignettes.py) : précision 59 % → 80 % (mémoire + adaptateur).
# Tout est aussi rangé en dataset au format YOLO (dataset/images, dataset/labels, data.yaml), prêt pour un
# entraînement complet du détecteur quand il y aura assez d'images.
# Exemples PUBLICS (Open Images V7, licence CC-BY, bench/encoder_open_images.py) : l'adaptateur s'entraîne AUSSI sur
# eux, avec un poids plus faible que tes corrections. La mémoire (niveau 2) reste faite de TES corrections seulement.
PUBLICS = PROJET / "dataset" / "open_images" / "signatures.pt"
CONFIG.setdefault("public_poids", 0.05)   # poids d'un exemple Open Images (1 = autant qu'une de tes corrections ; 0 = ignorés)
                                          # mesuré (bench/test_open_images.py) : 0,05 = ta pièce inchangée (79-80 %), autres pièces 73 → 79 %

class Apprentissage:
    DOSSIER = PROJET / "dataset"

    def __init__(self, verif, cerveau, dossier=None):
        self.v, self.cerveau = verif, cerveau
        self.DOSSIER = Path(dossier) if dossier else self.DOSSIER
        for d in ("images", "labels", "crops"):
            (self.DOSSIER / d).mkdir(parents=True, exist_ok=True)
        self.fichier_ex = self.DOSSIER / "exemples.jsonl"
        self.fichier_sig = self.DOSSIER / "signatures.pt"
        self.fichier_W = self.DOSSIER / "adaptateur.pt"
        self.fichier_classes = self.DOSSIER / "classes.json"
        self.ex = [json.loads(l) for l in self.fichier_ex.read_text().splitlines()] if self.fichier_ex.exists() else []
        sig = torch.load(self.fichier_sig) if self.fichier_sig.exists() else {"E": torch.zeros(0, 512), "V": torch.zeros(0, 512)}
        n = min(len(self.ex), len(sig["E"]))
        self.ex, self.E, self.V = self.ex[:n], sig["E"][:n].to(verif.dev), sig["V"][:n]
        self.classes = json.loads(self.fichier_classes.read_text()) if self.fichier_classes.exists() else []
        self.W, self.bilan = None, "pas encore entraîné"
        self.P = self._charger_publics()
        self.a_revoir, self._proposes = deque(maxlen=CONFIG["revoir_max"]), set()
        verif.apprentissage = self
        if len(self.ex) >= CONFIG["entrainement_min"] or self.P:
            self.entrainer()

    def _charger_publics(self):
        """(signatures, noms) des exemples Open Images d'entraînement (ceux mis de côté pour la mesure sont exclus)."""
        if not CONFIG["public_poids"] or not PUBLICS.exists():
            return None
        d = torch.load(PUBLICS)
        garde = [i for i, v in enumerate(d["val"]) if not v]
        return d["E"][garde].to(self.v.dev).float(), [d["noms"][i] for i in garde]

    # ---------- niveau 2 : mémoire visuelle
    @torch.no_grad()
    def souvenir(self, I):
        """Pour chaque signature : (nom, ressemblance) si elle ressemble fort à des exemples corrigés, sinon None."""
        if not len(self.E):
            return [None] * len(I)
        S = I @ self.E.T
        res = []
        for s in S:
            v, i = s.topk(min(CONFIG["memoire_k"], len(self.E)))
            votes = defaultdict(float)
            for vv, ii in zip(v.tolist(), i.tolist()):
                if vv >= CONFIG["memoire_seuil"]:
                    votes[self.ex[ii]["nom"]] += vv
            res.append((max(votes, key=votes.get), float(v[0])) if votes else None)
        return res

    def _classe_id(self, nom):
        if nom not in self.classes:
            self.classes.append(nom)
            self.fichier_classes.write_text(json.dumps(self.classes, ensure_ascii=False, indent=1))
            (self.DOSSIER / "data.yaml").write_text(
                f"path: {self.DOSSIER}\ntrain: images\nval: images\nnames:\n"
                + "".join(f"  {i}: {json.dumps(n, ensure_ascii=False)}\n" for i, n in enumerate(self.classes)))
        return self.classes.index(nom)

    def _enregistrer(self, ident, nom, e, vpe, boite, source):
        self.ex.append({"id": ident, "nom": nom, "boite": list(boite) if boite else None, "source": source,
                        "date": datetime.now().isoformat(timespec="seconds")})
        self.E = torch.cat([self.E, e[None]])
        self.V = torch.cat([self.V, vpe.float().cpu()[None]])
        with open(self.fichier_ex, "a") as f:
            f.write(json.dumps(self.ex[-1], ensure_ascii=False) + "\n")
        torch.save({"E": self.E.cpu(), "V": self.V}, self.fichier_sig)
        if len(self.ex) >= CONFIG["entrainement_min"] and len(self.ex) % CONFIG["entrainement_auto"] == 0:
            threading.Thread(target=self.entrainer, daemon=True).start()   # ré-entraîne tout seul, en arrière-plan

    def ajouter(self, nom, frame, boite, parts=None, autres=(), source="correction"):
        """Enregistre une correction. nom = nom français, ou "REJET" (= ce n'est pas un objet)."""
        crop = self.v.decouper(frame, boite, parts)
        e = self.v.encoder([crop])[0]
        try:
            vpe = self.cerveau.embedding_visuel(frame, boite) if nom != "REJET" else torch.zeros(512)
        except Exception:
            vpe = torch.zeros(512)
        ident = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        cv2.imwrite(str(self.DOSSIER / "crops" / f"{ident}.jpg"), cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
        if nom != "REJET":                                   # image complète + étiquettes au format YOLO
            H, W = frame.shape[:2]
            cv2.imwrite(str(self.DOSSIER / "images" / f"{ident}.jpg"), frame)
            lignes = []
            for n, (x1, y1, x2, y2) in [(nom, boite)] + list(autres):
                lignes.append(f"{self._classe_id(n)} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} "
                              f"{(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}")
            (self.DOSSIER / "labels" / f"{ident}.txt").write_text("\n".join(lignes) + "\n")
        self._enregistrer(ident, nom, e, vpe, boite, source)

    def ajouter_vignette(self, nom, crop_bgr, source):
        """Importe un exemple déjà découpé (sans image complète : sert à la mémoire et à l'adaptateur seulement)."""
        crop = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
        ident = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        cv2.imwrite(str(self.DOSSIER / "crops" / f"{ident}.jpg"), crop_bgr)
        self._enregistrer(ident, nom, self.v.encoder([crop])[0], torch.zeros(512), None, source)

    # ---------- file « à revoir » (apprentissage actif : il te montre ce dont il n'est pas sûr)
    def proposer(self, uid, frame, boite, parts, propositions):
        if uid in self._proposes or len(self.a_revoir) >= self.a_revoir.maxlen:
            return
        self._proposes.add(uid)
        ok, jpg = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
        self.a_revoir.append({"jpg": jpg, "boite": boite, "parts": parts, "propositions": propositions[:3]})

    # ---------- niveau 3 : entraînement
    def entrainer(self):
        t0 = time.time()
        n = self.v.n_classes
        idx, cibles = [], []
        for k, e in enumerate(self.ex):
            if e["nom"] == "REJET":
                idx.append(k); cibles.append(n)
            elif e["nom"] in self.cerveau.noms_fr:
                idx.append(k); cibles.append(self.cerveau.noms_fr.index(e["nom"]))
        Xp, yp = torch.zeros(0, 512, device=self.v.dev), []
        if self.P:                                           # exemples publics dont le nom est dans le vocabulaire
            ip = [i for i, nom in enumerate(self.P[1]) if nom in self.cerveau.noms_fr]
            Xp, yp = self.P[0][ip], [self.cerveau.noms_fr.index(self.P[1][i]) for i in ip]
        if len(idx) < CONFIG["entrainement_min"] and not yp:
            self.bilan = (f"{len(idx)} correction{'s' if len(idx) > 1 else ''} pour l'instant : "
                          f"il en faut au moins {CONFIG['entrainement_min']} pour m'entraîner")
            return self.bilan
        dev = Xp.device
        X = torch.cat([self.E[idx].float().to(dev), Xp])
        y = torch.tensor(cibles + yp, device=dev)
        poids = torch.cat([torch.ones(len(idx), device=dev), torch.full((len(yp),), float(CONFIG["public_poids"]), device=dev)])
        T = self.v.textes.float()
        Tc = torch.cat([T[:n], torch.nn.functional.normalize(T[n:].mean(0, keepdim=True), dim=-1)])  # rejets = 1 classe
        W = torch.zeros(512, 512, device=X.device, requires_grad=True)
        opt = torch.optim.Adam([W], lr=1e-3)
        with torch.enable_grad():
            for _ in range(CONFIG["entrainement_epoques"]):
                Z = torch.nn.functional.normalize(X + X @ W, dim=-1)
                erreurs = torch.nn.functional.cross_entropy(self.v.echelle * Z @ Tc.T, y, reduction="none")
                perte = (erreurs * poids).sum() / poids.sum() + CONFIG["entrainement_reg"] * (W ** 2).sum()
                opt.zero_grad(); perte.backward(); opt.step()
        self.W = W.detach()
        torch.save(self.W.cpu(), self.fichier_W)
        # signatures du détecteur : pour chaque objet corrigé au moins N fois, on mélange le texte et TES images
        protos = {}
        for nom in {e["nom"] for e in self.ex if e["nom"] != "REJET"}:
            ks = [k for k, e in enumerate(self.ex) if e["nom"] == nom and self.V[k].abs().sum() > 0]
            if len(ks) >= CONFIG["proto_min"] and nom in self.cerveau.noms_fr and not self.cerveau.est_perso[self.cerveau.noms_fr.index(nom)]:
                protos[nom] = torch.nn.functional.normalize(self.V[ks].mean(0), dim=-1)
        if protos.keys() != self.cerveau.prototypes.keys():
            self.cerveau.prototypes = protos
            self.cerveau.appliquer()
        noms = {e["nom"] for e in self.ex}
        publics = f" + {len(yp)} exemples Open Images (poids {CONFIG['public_poids']})" if yp else ""
        self.bilan = (f"entraîné sur {len(idx)} exemples ({len(noms)} noms){publics} en {time.time() - t0:.1f} s, "
                      f"{len(protos)} objet(s) adapté(s) dans le détecteur")
        return self.bilan

apprentissage = Apprentissage(verificateur, cerveau) if verificateur else None
if apprentissage:
    print(f"Apprentissage : {len(apprentissage.ex)} exemples enregistrés | {apprentissage.bilan}")

In [ ]:
# 5) CAMÉRAS : la webcam du PC et/ou l'iPhone (app DroidCam), chacune dans son thread
# On traite toujours l'image la plus récente (latence minimale). Une caméra qui décroche (Wi-Fi coupé, app fermée,
# iPhone verrouillé) se reconnecte toute seule ; les autres continuent pendant ce temps.
# Mesuré avec le vrai iPhone (DroidCam, Wi-Fi de la box) : 1280x720, ~21 images/s, 4 ms de réseau.
import socket
from concurrent.futures import ThreadPoolExecutor

CONFIG["cameras"] = {
    "webcam": 0,                  # /dev/video0
    "iphone": "auto",             # "auto" = DroidCam trouvé tout seul (partage de connexion de l'iPhone OU même Wi-Fi) ;
                                  # sinon une adresse fixe, ex. "http://192.0.2.10:4747/video" ; None = pas d'iPhone
}
CONFIG["rotations"] = {           # degrés dans le sens des aiguilles d'une montre (0, 90, 180, 270)
    "iphone": 90,                 # iPhone tenu DEBOUT : DroidCam envoie l'image couchée → on la redresse
}                                 # (iPhone posé en paysage sur un support → 0)
DROIDCAM_PORT = 4747
DROIDCAM_MEMO = REGLAGES / "iphone_adresse.txt"   # dernière adresse trouvée : essayée en premier la fois suivante
ROTATIONS = {90: cv2.ROTATE_90_CLOCKWISE, 180: cv2.ROTATE_180, 270: cv2.ROTATE_90_COUNTERCLOCKWISE}
# Un flux réseau qui ne répond plus est abandonné au bout de 3 s (au lieu de ~30 s) pour tenter une reconnexion
os.environ.setdefault("OPENCV_FFMPEG_CAPTURE_OPTIONS", "timeout;3000000")

def _passerelle():
    """Passerelle de la route par défaut (la box, ou l'iPhone en partage de connexion)."""
    for ligne in Path("/proc/net/route").read_text().splitlines()[1:]:
        champs = ligne.split()
        if champs[1] == "00000000":
            return ".".join(str(int(champs[2][i:i + 2], 16)) for i in (6, 4, 2, 0))
    return None

def _droidcam_ouvert(ip, delai=0.4):
    try:
        with socket.create_connection((ip, DROIDCAM_PORT), timeout=delai):
            return True
    except OSError:
        return False

def trouver_iphone():
    """Adresse IP de l'iPhone qui fait tourner DroidCam, ou None.
    1) partage de connexion : l'iPhone EST la passerelle (172.20.10.x) ; 2) la dernière adresse trouvée ;
    3) sinon on demande au port 4747 de chaque appareil du réseau local (254 essais en parallèle, ~1 s)."""
    gw = _passerelle()
    if gw and gw.startswith("172.20.10.") and _droidcam_ouvert(gw):
        return gw
    if DROIDCAM_MEMO.exists() and _droidcam_ouvert(DROIDCAM_MEMO.read_text().strip()):
        return DROIDCAM_MEMO.read_text().strip()
    if not gw:
        return None
    reseau = gw.rsplit(".", 1)[0]
    with ThreadPoolExecutor(64) as pool:
        ips = [f"{reseau}.{i}" for i in range(1, 255)]
        trouve = next((ip for ip, ok in zip(ips, pool.map(_droidcam_ouvert, ips)) if ok), None)
    if trouve:
        DROIDCAM_MEMO.write_text(trouve)
    return trouve


class Camera:
    """Lit une caméra en continu : un numéro = webcam du PC, un texte = adresse réseau (iPhone DroidCam, caméra IP...),
    "auto" = iPhone trouvé tout seul. L'image est redressée selon CONFIG["rotations"]."""
    def __init__(self, nom, source, w=CONFIG["largeur"], h=CONFIG["hauteur"]):
        self.nom, self.source, self.w, self.h = nom, source, w, h
        self.rotation = ROTATIONS.get(CONFIG["rotations"].get(nom, 0))
        self.frame, self.t_image, self.cap, self.adresse = None, 0.0, None, None
        self.actif, self.reconnexions = True, 0
        self.lock = threading.Lock()
        self.thread = threading.Thread(target=self._boucle, daemon=True)
        self.thread.start()

    def _ouvrir(self):
        if isinstance(self.source, int):
            cap = cv2.VideoCapture(self.source, cv2.CAP_V4L2)
            cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))  # MJPG = 30 FPS en HD sur la plupart des webcams
            cap.set(cv2.CAP_PROP_FRAME_WIDTH, self.w)
            cap.set(cv2.CAP_PROP_FRAME_HEIGHT, self.h)
        else:
            if self.source == "auto":                    # cherché à chaque (re)connexion : le réseau a pu changer
                ip = trouver_iphone()
                self.adresse = ip and f"http://{ip}:{DROIDCAM_PORT}/video"
            else:
                self.adresse = self.source
            if not self.adresse:
                return None
            cap = cv2.VideoCapture(self.adresse, cv2.CAP_FFMPEG)
        cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
        if cap.isOpened():
            return cap
        cap.release()
        return None

    def _boucle(self):
        while self.actif:
            if self.cap is None:
                self.cap = self._ouvrir()
                if self.cap is None:                     # pas encore là (app fermée, Wi-Fi...) : on réessaie
                    time.sleep(2)
                    continue
            ok, f = self.cap.read()
            if ok and f is not None:
                if self.rotation is not None:
                    f = cv2.rotate(f, self.rotation)
                with self.lock:
                    self.frame, self.t_image = f, time.time()
            else:                                        # flux coupé → on referme et on se reconnecte
                self.cap.release()
                self.cap, self.reconnexions = None, self.reconnexions + 1
                time.sleep(1)

    @property
    def en_ligne(self):
        return time.time() - self.t_image < 3

    def lire(self):
        """L'image la plus récente, ou None si la caméra ne donne plus rien depuis 3 s."""
        with self.lock:
            return self.frame.copy() if self.frame is not None and self.en_ligne else None

    def attendre(self, delai=8):
        """Attend une première image correcte (une webcam renvoie du noir pendant le réglage de l'exposition)."""
        t = time.time()
        while time.time() - t < delai:
            f = self.lire()
            if f is not None and f.mean() > 15:
                return True
            time.sleep(0.05)
        return self.lire() is not None

    def stop(self):
        self.actif = False
        self.thread.join(timeout=4)
        if self.cap is not None:
            self.cap.release()


def _cadrage(forme, w, h):
    """Échelle et décalage pour mettre une image de cette forme dans w x h sans la déformer."""
    k = min(w / forme[1], h / forme[0])
    nw, nh = int(forme[1] * k), int(forme[0] * k)
    return k, nw, nh, (w - nw) // 2, (h - nh) // 2

def adapter(f, w, h):
    """Met une image au format de la fenêtre sans la déformer (bandes noires si besoin)."""
    if f.shape[1] == w and f.shape[0] == h:
        return f
    k, nw, nh, x, y = _cadrage(f.shape, w, h)
    sortie = np.zeros((h, w, 3), np.uint8)
    sortie[y:y + nh, x:x + nw] = cv2.resize(f, (nw, nh))
    return sortie

def recadrer(dets, forme, w, h):
    """Ramène les détections faites sur l'image d'ORIGINE dans l'image adaptée à la fenêtre.
    (On analyse l'image d'origine : un iPhone tenu debout, réduit avec des bandes noires, rend les personnes
    trop petites pour YOLOE — mesuré : confiance « personne » 0,35 au lieu de 0,92.)"""
    if forme[1] == w and forme[0] == h:
        return dets
    k, _, _, x, y = _cadrage(forme, w, h)
    for d in dets:
        x1, y1, x2, y2 = d["b"]
        d["b"] = (int(x1 * k) + x, int(y1 * k) + y, int(x2 * k) + x, int(y2 * k) + y)
        if d["poly"]:
            d["poly"] = [(p * k + [x, y]).astype(np.int32) for p in d["poly"]]
    return dets


# Test rapide de chaque caméra configurée
for _nom, _source in CONFIG["cameras"].items():
    if _source is None:
        continue
    _t = time.time()
    _cam = Camera(_nom, _source)
    if _cam.attendre(8 if isinstance(_source, int) else 6):
        _f = _cam.lire()
        print(f"Caméra {_nom} OK — {_f.shape[1]}x{_f.shape[0]} | prête en {time.time() - _t:.1f}s | "
              f"luminosité {_f.mean():.0f}/255" + (f" | {_cam.adresse}" if _cam.adresse else ""))
    else:
        print(f"Caméra {_nom} : aucune image" + (" — DroidCam est-il ouvert sur l'iPhone ?" if _source == "auto" else "")
              + " Elle sera réessayée au lancement.")
    _cam.stop()

In [ ]:
# 6) Moteur graphique : HUD, modes de vision, traînées, contours exacts (segmentation)
from ultralytics.utils import ops

POLICE = cv2.FONT_HERSHEY_DUPLEX
CYAN, BLANC, GRIS, VERT, ROUGE, ORANGE, OR = (255, 255, 0), (240, 240, 240), (130, 130, 130), (80, 255, 80), (60, 60, 255), (0, 165, 255), (0, 215, 255)
MODES = {1: "HUD", 2: "SPOTLIGHT", 3: "HEATMAP", 4: "VISION NOCTURNE"}

def couleur(nom, perso=False):
    """Couleur néon stable pour chaque nom d'objet (or pour tes objets perso)."""
    if perso:
        return OR
    if nom == "personne":
        return CYAN
    h = sum(map(ord, nom)) * 37 % 180
    return tuple(int(c) for c in cv2.cvtColor(np.uint8([[[h, 210, 255]]]), cv2.COLOR_HSV2BGR)[0, 0])

def texte(img, s, org, echelle=0.5, couleur=BLANC, ep=1):
    s = sans_accents(s)
    x, y = org
    cv2.putText(img, s, (x + 1, y + 1), POLICE, echelle, (0, 0, 0), ep + 2, cv2.LINE_AA)
    cv2.putText(img, s, (x, y), POLICE, echelle, couleur, ep, cv2.LINE_AA)

def cadre_coins(img, x1, y1, x2, y2, c, ep=2):
    """Cadre façon viseur : seuls les coins sont tracés + contour fin."""
    L = max(10, int(min(x2 - x1, y2 - y1) * 0.22))
    cv2.rectangle(img, (x1, y1), (x2, y2), c, 1, cv2.LINE_AA)
    for (px, py, dx, dy) in ((x1, y1, 1, 1), (x2, y1, -1, 1), (x1, y2, 1, -1), (x2, y2, -1, -1)):
        cv2.line(img, (px, py), (px + dx * L, py), c, ep + 1, cv2.LINE_AA)
        cv2.line(img, (px, py), (px, py + dy * L), c, ep + 1, cv2.LINE_AA)

def etiquette(img, x, y, s, c):
    s = sans_accents(s)
    (tw, th), _ = cv2.getTextSize(s, POLICE, 0.5, 1)
    y = max(y, th + 8)
    cv2.rectangle(img, (x, y - th - 8), (x + tw + 10, y), c, -1)
    cv2.putText(img, s, (x + 5, y - 5), POLICE, 0.5, (0, 0, 0), 1, cv2.LINE_AA)

def contours_masques(masks):
    """Pour chaque objet : la liste de ses morceaux de contour (en pixels de l'image d'origine).
    (masks.xy d'ultralytics relie les morceaux entre eux par des traits → grandes lignes parasites.)"""
    if masks is None:
        return None
    data = masks.data.byte().cpu().numpy()
    resultat = []
    for m in data:
        cs = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]
        resultat.append([ops.scale_coords(data.shape[1:], c.reshape(-1, 2).astype(np.float32), masks.orig_shape).astype(np.int32)
                         for c in cs if len(c) >= 3 and cv2.contourArea(c) > 15])
    return resultat

def contour(img, parts, c, alpha=0.18):
    """Contour exact de l'objet (masque de segmentation) + léger remplissage."""
    if not parts:
        return
    x, y, w, h = cv2.boundingRect(np.concatenate(parts))
    x2, y2 = min(x + w, img.shape[1]), min(y + h, img.shape[0])
    if x2 <= x or y2 <= y:
        return
    roi = img[y:y2, x:x2]
    calque = roi.copy()
    cv2.fillPoly(calque, [p - [x, y] for p in parts], c)
    cv2.addWeighted(calque, alpha, roi, 1 - alpha, 0, dst=roi)
    cv2.polylines(img, parts, True, c, 1, cv2.LINE_AA)

def mode_spotlight(frame, contours, boites):
    """Fond sombre et désaturé : seuls les objets (leur forme exacte) restent éclairés."""
    gris = cv2.cvtColor(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), cv2.COLOR_GRAY2BGR)
    out = (gris * 0.22).astype(np.uint8)
    masque = np.zeros(frame.shape[:2], np.uint8)
    for parts, (x1, y1, x2, y2) in zip(contours, boites):
        if parts:
            cv2.fillPoly(masque, parts, 255)
        else:
            masque[y1:y2, x1:x2] = 255
    masque = cv2.GaussianBlur(masque, (0, 0), 3)[..., None] / 255.0
    return (frame * masque + out * (1 - masque)).astype(np.uint8)

def mode_nocturne(frame):
    """Effet jumelles de vision nocturne : vert, grain, lignes de balayage, vignette."""
    g = cv2.equalizeHist(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)).astype(np.float32)
    g += np.random.normal(0, 12, g.shape).astype(np.float32)
    g[::3] *= 0.75
    h, w = g.shape
    yy, xx = np.ogrid[:h, :w]
    vignette = 1 - 0.65 * (((xx - w / 2) / (w / 2)) ** 2 + ((yy - h / 2) / (h / 2)) ** 2)
    g = np.clip(g * np.clip(vignette, 0, 1), 0, 255)
    out = np.zeros_like(frame)
    out[..., 1] = g.astype(np.uint8)
    out[..., 0] = (g * 0.25).astype(np.uint8)
    return out

class Heatmap:
    """Mémoire des zones où les objets apparaissent (s'efface lentement)."""
    def __init__(self, h, w):
        self.carte = np.zeros((h, w), np.float32)
    def maj(self, boites):
        self.carte *= 0.985
        for (x1, y1, x2, y2) in boites:
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
            cv2.ellipse(self.carte, (cx, cy), (max(5, (x2 - x1) // 3), max(5, (y2 - y1) // 3)), 0, 0, 360, 1.0, -1)
    def rendu(self, frame):
        m = cv2.GaussianBlur(self.carte, (0, 0), 25)
        m = np.clip(m / (m.max() + 1e-6) * 255, 0, 255).astype(np.uint8)
        return cv2.addWeighted((frame * 0.45).astype(np.uint8), 1.0, cv2.applyColorMap(m, cv2.COLORMAP_INFERNO), 0.65, 0)

def dessiner_trainees(img, trainees, couleurs_id):
    for tid, pts in trainees.items():
        p = list(pts)
        for i in range(1, len(p)):
            cv2.line(img, p[i - 1], p[i], couleurs_id.get(tid, CYAN), max(1, int(4 * i / len(p))), cv2.LINE_AA)

def viseur_central(img, t):
    h, w = img.shape[:2]
    cx, cy = w // 2, h // 2
    r = 18 + int(4 * np.sin(t * 4))
    cv2.circle(img, (cx, cy), r, CYAN, 1, cv2.LINE_AA)
    for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
        cv2.line(img, (cx + dx * (r + 4), cy + dy * (r + 4)), (cx + dx * (r + 14), cy + dy * (r + 14)), CYAN, 1, cv2.LINE_AA)

def verrouillage(img, x1, y1, x2, y2, t):
    """Effet 'cible verrouillée' pour l'objet recherché : anneaux tournants + ligne depuis le centre."""
    cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
    r = int(max(x2 - x1, y2 - y1) * 0.6) + 10
    a = int(t * 180) % 360
    for k in range(3):
        cv2.ellipse(img, (cx, cy), (r, r), 0, a + k * 120, a + k * 120 + 70, ROUGE, 2, cv2.LINE_AA)
    cv2.circle(img, (cx, cy), 4, ROUGE, -1)
    h, w = img.shape[:2]
    cv2.line(img, (w // 2, h // 2), (cx, cy), ROUGE, 1, cv2.LINE_AA)
    texte(img, "CIBLE", (cx - 25, cy - r - 8), 0.55, ROUGE, 1)

def bandeau(img, s, c=CYAN, y=None):
    """Bandeau central semi-transparent (consignes, saisie de texte, réponses)."""
    h, w = img.shape[:2]
    y = y or h - 70
    calque = img.copy()
    cv2.rectangle(calque, (0, y - 34), (w, y + 14), (0, 0, 0), -1)
    cv2.addWeighted(calque, 0.65, img, 0.35, 0, dst=img)
    texte(img, s, (20, y), 0.75, c, 1)

print("Moteur graphique prêt — modes :", ", ".join(f"{k}={v}" for k, v in MODES.items()))

In [ ]:
# 6b) ENREGISTREMENT de l'écran (touche R) en H.264 : la vidéo se lit partout (iPhone, WhatsApp, VLC...)
import av

class VideoH264:
    """Même usage que cv2.VideoWriter (write / release), mais en H.264 et à VITESSE RÉELLE : quel que soit le rythme
    de la fenêtre (8 à 25 images/s), les images sont répétées ou sautées pour garder 15 images/s.
    L'encodage se fait dans un thread : la fenêtre ne ralentit pas (mesuré : ~4 ms par image)."""
    FPS = 15

    def __init__(self, chemin, largeur, hauteur):
        self.chemin, self.t0, self.n = chemin, None, 0
        self.q = queue.Queue(maxsize=30)
        self.sortie = av.open(str(chemin), "w")
        self.flux = self.sortie.add_stream("libx264", rate=self.FPS)
        self.flux.width, self.flux.height, self.flux.pix_fmt = largeur, hauteur, "yuv420p"
        self.flux.options = {"crf": "23", "preset": "veryfast"}
        self.thread = threading.Thread(target=self._boucle, daemon=True)
        self.thread.start()

    def write(self, img):
        t = time.time()
        self.t0 = self.t0 or t
        n = round((t - self.t0) * self.FPS) + 1 - self.n     # nb d'images à écrire pour rester à l'heure
        if n <= 0:
            return
        self.n += n
        try:
            self.q.put_nowait((img.copy(), n))
        except queue.Full:                                    # encodeur en retard : on saute plutôt que ralentir
            pass

    def _boucle(self):
        while (item := self.q.get()) is not None:
            img, n = item
            image = av.VideoFrame.from_ndarray(img, format="bgr24")
            for _ in range(n):
                for paquet in self.flux.encode(image):
                    self.sortie.mux(paquet)
        for paquet in self.flux.encode():
            self.sortie.mux(paquet)
        self.sortie.close()

    def release(self):
        self.q.put(None)
        self.thread.join(timeout=30)

print("Enregistrement vidéo (touche R) : H.264,", VideoH264.FPS, "images/s, vitesse réelle")

In [ ]:
# 7) Stabilisation des noms, Journal, MÉMOIRE des objets, Mission
class Stabilisateur:
    """Chaque objet suivi vote sur ses N dernières images → le nom affiché ne 'saute' plus
    (fini téléphone / télécommande / clé USB qui alternent)."""
    def __init__(self):
        self.votes = defaultdict(lambda: deque(maxlen=CONFIG["vote"]))
        self.age = Counter()
    def maj(self, uid, c, conf):
        self.votes[uid].append((c, conf)); self.age[uid] += 1
        score = defaultdict(float)
        for cc, s in self.votes[uid]:
            score[cc] += s
        meilleur = max(score, key=score.get)
        return meilleur, score[meilleur] / len(self.votes[uid])   # classe stable + confiance moyenne
    def oublier(self, actifs):
        for uid in [u for u in self.votes if u not in actifs]:
            del self.votes[uid]


class Journal:
    """Note chaque NOUVEL objet (suivi depuis ≥ 3 images, pour ignorer les fausses alertes)."""
    def __init__(self):
        self.vus, self.lignes, self.total = set(), deque(maxlen=9), Counter()
        self.fichier = LOGS / f"session_{datetime.now():%Y%m%d_%H%M%S}.csv"
        self._f = open(self.fichier, "w", newline="")
        self._w = csv.writer(self._f)
        self._w.writerow(["heure", "evenement", "id", "objet", "confiance"])

    def ajouter(self, s, couleur=BLANC):
        self.lignes.appendleft((f"{datetime.now():%H:%M:%S}  {s}", couleur))

    def observer(self, uid, nom, conf, age, perso):
        if uid in self.vus or age < 3:
            return
        self.vus.add(uid); self.total[nom] += 1
        self.ajouter(f"+ {nom} #{uid % 10_000_000}", couleur(nom, perso))
        self._w.writerow([datetime.now().isoformat(timespec="seconds"), "apparition", uid, nom, f"{conf:.2f}"])

    def fermer(self):
        self._f.close()


def il_y_a(t):
    d = time.time() - t
    if d < 5:    return "à l'instant"
    if d < 60:   return f"il y a {int(d)} secondes"
    if d < 3600: return f"il y a {int(d // 60)} minute{'s' if d >= 120 else ''}"
    jour = datetime.fromtimestamp(t)
    if jour.date() == datetime.now().date():
        return f"aujourd'hui à {jour:%Hh%M}"
    return f"le {jour:%d/%m} à {jour:%Hh%M}"


class Memoire:
    """Se souvient OÙ et QUAND chaque objet a été vu pour la dernière fois (gardé entre les sessions)."""
    FICHIER = MEMOIRE / "ou_sont_les_objets.json"

    def __init__(self):
        self.d = json.loads(self.FICHIER.read_text()) if self.FICHIER.exists() else {}
        self._dernier, self._sale, self._t_sauve = {}, False, time.time()

    @staticmethod
    def position(boite, W, H, voisins):
        x1, y1, x2, y2 = boite
        cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
        horiz = "à gauche" if cx < 0.36 else "à droite" if cx > 0.64 else "au centre"
        vert = " en haut" if cy < 0.33 else " en bas" if cy > 0.7 else ""
        s = f"{horiz}{vert} de l'image"
        if voisins:   # l'objet le plus proche donne un repère concret
            proche = min(voisins, key=lambda v: (v[1][0] + v[1][2] - x1 - x2) ** 2 + (v[1][1] + v[1][3] - y1 - y2) ** 2)
            s += f", près de : {proche[0]}"
        return s

    def maj(self, objets, frame, camera=None):
        """objets = [(nom, boite)] stables de l'image courante ; camera = son nom quand il y en a plusieurs."""
        H, W = frame.shape[:2]
        t = time.time()
        for nom, boite in objets:
            if t - self._dernier.get(nom, 0) < 2:      # une mise à jour toutes les 2 s par objet suffit
                continue
            self._dernier[nom] = t
            voisins = [(n, b) for n, b in objets if n not in (nom, "personne")]
            x1, y1, x2, y2 = boite
            m = int(0.15 * max(x2 - x1, y2 - y1))
            vignette = VIGNETTES / f"vu_{sans_accents(nom).replace(' ', '_')}.jpg"
            cv2.imwrite(str(vignette), frame[max(0, y1 - m):y2 + m, max(0, x1 - m):x2 + m])
            e = self.d.get(nom, {"n": 0})
            ou = self.position(boite, W, H, voisins) + (f", sur la caméra {camera}" if camera else "")
            self.d[nom] = {"t": t, "ou": ou, "img": str(vignette), "n": e["n"] + 1}
            self._sale = True
        if self._sale and t - self._t_sauve > 5:
            self.sauver()

    def sauver(self):
        self.FICHIER.write_text(json.dumps(self.d, ensure_ascii=False, indent=1))
        self._sale, self._t_sauve = False, time.time()

    def repondre(self, nom, visibles):
        if nom in visibles:
            return f"Je vois {nom} en ce moment, {self.d.get(nom, {}).get('ou', 'devant moi')}."
        e = self.d.get(nom)
        if not e:
            return f"Je n'ai jamais vu {nom}. Montre-le-moi une fois et je m'en souviendrai."
        return f"J'ai vu {nom} {il_y_a(e['t'])}, {e['ou']}."


OBJETS_MISSION = [VOCAB[k] for k in ["cup", "bottle", "cell phone", "book", "pen", "key", "wallet", "spoon",
                  "plate", "computer mouse", "remote control", "scissors", "backpack", "chair", "eyeglasses",
                  "wristwatch", "banana", "orange", "toothbrush", "comb", "charger", "usb flash drive", "coin"]]

class Mission:
    """Le système demande un objet : montre-le à la caméra et garde-le visible 1 seconde."""
    DUREE_MAX, MAINTIEN = 30.0, 1.0

    def __init__(self, journal):
        self.journal, self.score, self.reussies = journal, 0, 0
        self.cible, self.debut, self.vu_depuis, self.flash = None, 0.0, None, 0.0

    def nouvelle(self):
        self.cible = random.choice([c for c in OBJETS_MISSION if c != self.cible])
        self.debut, self.vu_depuis = time.time(), None
        self.journal.ajouter(f"MISSION: trouve {self.cible}", ORANGE)
        return self.cible

    def maj(self, visibles):
        if self.cible is None:
            return None
        t = time.time()
        if self.cible in visibles:
            self.vu_depuis = self.vu_depuis or t
            if t - self.vu_depuis >= self.MAINTIEN:
                gain = int(100 + 10 * max(0, self.DUREE_MAX - (t - self.debut)))
                self.score += gain; self.reussies += 1; self.flash = t
                self.journal.ajouter(f"REUSSI {self.cible} +{gain}", VERT)
                ok = self.cible
                self.nouvelle()
                return f"Bravo ! {ok} trouvé, plus {gain} points."
        else:
            self.vu_depuis = None
            if t - self.debut > self.DUREE_MAX:
                self.journal.ajouter(f"ECHEC {self.cible}", ROUGE)
                return f"Temps écoulé. Nouvelle mission : {self.nouvelle()}."
        return None

    def progression(self):
        return 0.0 if self.vu_depuis is None else min(1.0, (time.time() - self.vu_depuis) / self.MAINTIEN)

    def restant(self):
        return max(0.0, self.DUREE_MAX - (time.time() - self.debut))

print("Stabilisateur, Journal, Mémoire, Mission prêts")

In [ ]:
# 7b) PERCEPTION : tout le travail de « voir » au même endroit (utilisé par la fenêtre ET par le banc d'essai)
def decider(votes, v, c_yolo):
    """1) Si l'objet ressemble fort à une de TES corrections → on la suit (niveau 2).
    2) Sinon le vérificateur PROPOSE (il regarde l'objet de près), YOLOE DÉPARTAGE (ses votes sur les dernières images).
    Retourne (classe, part du meilleur nom, étiquette affichée, incertain, rejeté, 3 meilleures propositions)."""
    n = len(cerveau.noms_fr)
    py = np.zeros(n)
    for c, s in votes:
        py[c] += s
    py /= max(py.max(), 1e-6)                            # 1 pour le choix de YOLOE, moins pour les autres
    if v is None:                                        # pas encore vérifié : on garde l'avis de YOLOE
        c = int(py.argmax())
        return c, 1.0, cerveau.nom(c), False, False, [cerveau.nom(c)]
    souvenir = v.get("memoire")
    if souvenir:                                         # « tu m'as déjà dit ce que c'était »
        nom, ressemblance = souvenir
        if nom == "REJET":
            return None, 0.0, "", False, True, []
        if nom in cerveau.noms_fr:
            c = cerveau.noms_fr.index(nom)
            return c, ressemblance, nom, False, False, [nom]
    p = v["p"]
    pc, rejet = p[:n].copy(), float(p[n:].sum())
    i_pers = cerveau.textes_en.index("person")
    if c_yolo != i_pers:
        pc[i_pers] = 0                                   # un vêtement porté n'est pas « une personne »
    pc /= max(pc.sum(), 1e-6)
    top_v = float(pc.max())
    if rejet > CONFIG["verif_rejet"] and top_v < CONFIG["verif_top_rejet"]:   # « c'est une main / du flou / un mur »
        return None, 0.0, "", False, True, []
    if top_v < CONFIG["verif_perdu"]:                   # le vérificateur ne reconnaît rien : presque toujours une erreur
        return None, 0.0, "", False, True, []
    s = pc * ((1 - CONFIG["verif_alpha"]) + CONFIG["verif_alpha"] * py)
    s /= s.sum()
    ordre = np.argsort(-s)[:3]
    c1, c2 = ordre[:2]
    nom1, nom2 = cerveau.nom(c1), cerveau.nom(c2)
    incertain = s[c2] > CONFIG["verif_doute"] * s[c1]   # le 2e nom est presque aussi probable
    etiquette = nom1
    if incertain:
        cat = CATEGORIE_DE.get(nom1)
        etiquette = f"{cat} ({nom1} ?)" if cat and cat == CATEGORIE_DE.get(nom2) else f"{nom1} ?"
    return int(c1), float(s[c1]), etiquette, incertain, False, [cerveau.nom(i) for i in ordre]


class Perception:
    """Image → liste d'objets fiables : suivi (BoT-SORT) + vote + vérification + mémoire + hystérésis.
    Une Perception par caméra : chacune garde son propre tracker, et ses numéros d'objets commencent à `base`
    (la webcam à 0, l'iPhone à 10 000 000...) pour que deux caméras ne se mélangent jamais."""
    def __init__(self, base=0):
        self.stab = Stabilisateur()
        self.gen, self.offset, self.max_uid = None, base, base
        self.ema, self.vus, self.derniere_verif = {}, set(), {}
        self.rejets, self.saturations = 0, 0
        self.predictor = None                           # le tracker de CETTE caméra (YOLOE est partagé)

    def reverifier(self, uid):
        """Après une correction : on revérifie cet objet tout de suite (il suivra la correction)."""
        self.derniere_verif.pop(uid, None)
        if verificateur:
            verificateur.res.pop(uid, None)

    def analyser(self, frame, hd=False):
        H, W = frame.shape[:2]
        if cerveau.generation != self.gen:              # classes changées → tracker et vérificateur repartent
            self.gen, self.offset, self.predictor = cerveau.generation, self.max_uid, None
            if verificateur:
                verificateur.preparer(cerveau)
        model.predictor = self.predictor                # on remet le tracker de cette caméra avant de suivre
        try:
            res = model.track(frame, persist=True, tracker=str(TRACKER), conf=CONFIG["conf_suivi"],
                              imgsz=CONFIG["imgsz_hd"] if hd else CONFIG["imgsz"],
                              quantize=CONFIG["quantize"], device=CONFIG["device"], verbose=False)[0]
        except torch.OutOfMemoryError:                  # carte graphique pleine un instant (Qwen + Whisper...) :
            self.predictor = model.predictor            # on saute CETTE image au lieu de tout arrêter
            self.saturations += 1
            torch.cuda.empty_cache()
            if self.saturations in (1, 10, 100):
                print(f"Perception : carte graphique pleine, {self.saturations} image(s) sautée(s)", flush=True)
            return []
        self.predictor = model.predictor
        dets = []
        if res.boxes is None or not len(res.boxes) or res.boxes.id is None:
            return dets
        n = len(res.boxes)
        xyxy = res.boxes.xyxy.int().cpu().numpy()
        polys = contours_masques(res.masks) or [None] * n
        for b, c, s, i, poly in zip(xyxy, res.boxes.cls.int().tolist(), res.boxes.conf.tolist(),
                                    res.boxes.id.int().tolist(), polys):
            uid = i + self.offset
            self.max_uid = max(self.max_uid, uid)
            c_y, _ = self.stab.maj(uid, c, s)
            age = self.stab.age[uid]
            e = self.ema[uid] = s if uid not in self.ema else 0.7 * self.ema[uid] + 0.3 * s
            seuil = CONFIG["conf"] * (0.6 if uid in self.vus else 1.0)    # hystérésis : apparaître est plus dur que rester
            if age < CONFIG["age_min"] or e < seuil:
                continue
            boite = tuple(int(v) for v in np.clip(b, 0, [W - 1, H - 1, W - 1, H - 1]))
            perso = cerveau.est_perso[c_y]
            personne = cerveau.textes_en[c_y] == "person"
            souvenir = False
            if perso or personne:                       # YOLOE est très fiable sur les personnes et tes objets appris
                c_f, etiq, incertain, verifie, props = c_y, cerveau.nom(c_y), False, False, [cerveau.nom(c_y)]
            else:
                v = None
                if verificateur:
                    if age - self.derniere_verif.get(uid, -999) >= 45:   # 1re vérif tout de suite, puis toutes les ~1,5 s
                        verificateur.demander(uid, frame, boite, poly)
                        self.derniere_verif[uid] = age
                    v = verificateur.res.get(uid)
                    if v and v["gen"] != self.gen:
                        v = None
                    if v is None and age < CONFIG["age_min"] + 6:   # on attend le verdict (~0,2 s) avant d'afficher
                        continue
                c_f, _, etiq, incertain, rejete, props = decider(self.stab.votes[uid], v, c_y)
                if rejete:
                    self.rejets += 1
                    continue
                verifie = v is not None
                souvenir = bool(v and v.get("memoire"))
                if incertain and verifie and apprentissage:  # apprentissage actif : il te le montrera (touche E)
                    apprentissage.proposer(uid, frame, boite, poly, props)
            self.vus.add(uid)
            dets.append({"b": boite, "nom": cerveau.nom(c_f), "etiquette": etiq, "perso": cerveau.est_perso[c_f],
                         "conf": e, "uid": uid, "poly": poly, "age": age, "incertain": incertain, "verifie": verifie,
                         "souvenir": souvenir, "propositions": props})
        return dets

print("Perception prête (suivi BoT-SORT par caméra + vote + vérification + mémoire de tes corrections + hystérésis)")

In [ ]:
# 7c) SURVEILLANCE — zones, règles, événements (clip + visage) et alertes sur ton portable (Telegram)
# G = armer / désarmer (10 s pour sortir de la pièce)   Z = dessiner une zone (clics, Entrée, puis son nom)
# Chaque alerte → dossier evenements/AAAAMMJJ_HHMMSS_regle/ : image.jpg (le moment exact), clip.mp4 (10 s avant
# + 10 s après), visage.jpg (le visage le plus net vu pendant l'événement, s'il y en a un), info.json.
import av
import requests

EVENEMENTS = PROJET / "evenements"
EVENEMENTS.mkdir(parents=True, exist_ok=True)
ZONES_FICHIER = REGLAGES / "zones.json"
TELEGRAM_FICHIER = REGLAGES / "telegram.json"     # {"token": "123456:ABC...", "chat_id": null} — chat_id se remplit tout seul

CONFIG.update({
    "surv_avant_s": 10,           # secondes gardées AVANT le déclenchement
    "surv_apres_s": 10,           # secondes enregistrées APRÈS (prolongé si ça continue)
    "surv_fps": 10,               # images par seconde du clip (suffisant pour voir, léger à envoyer)
    "surv_largeur": 960,          # largeur du clip (px)
    "surv_delai": 10,             # après G : délai avant que la surveillance soit active (le temps de sortir)
    "surv_auto": None,            # ex. ("23:00", "06:00") → armée automatiquement la nuit
    "visage_seuil": 0.80,         # confiance minimum de YuNet pour garder un visage
})

ANIMAUX = list(CATEGORIES["animal"].values())
OBJETS_PRECIEUX = [VOCAB[k] for k in ("cell phone", "laptop", "wallet", "tablet computer", "key")]

# Les RÈGLES (actives seulement quand la surveillance est armée). Tu peux en ajouter / modifier ici :
#   type "presence"    : un de ces objets reste dans la zone pendant `duree` secondes → alerte
#   type "disparition" : un objet resté `present` s dans la zone n'y est plus depuis `absent` s → alerte
#   zones : "*" = toutes les zones dessinées (touche Z), ou toute l'image s'il n'y en a aucune ; ou une liste de noms
#   horaires : None = tout le temps, ou ("23:00", "06:00")      pause : délai minimum entre deux alertes
REGLES = [
    {"nom": "intrus", "type": "presence", "objets": ["personne"], "zones": "*", "duree": 1.5, "pause": 60, "horaires": None},
    {"nom": "animal", "type": "presence", "objets": ANIMAUX, "zones": "*", "duree": 2.0, "pause": 120, "horaires": None},
    {"nom": "objet disparu", "type": "disparition", "objets": OBJETS_PRECIEUX, "zones": "*", "present": 5, "absent": 5,
     "pause": 30, "horaires": None},
]


def dans_horaires(h, maintenant=None):
    if not h:
        return True
    m = (maintenant or datetime.now()).strftime("%H:%M")
    a, b = h
    return a <= m < b if a <= b else m >= a or m < b


class Zones:
    """Zones dessinées sur l'image d'UNE caméra (coordonnées en fraction de l'image : elles suivent un changement
    de résolution). La webcam garde config/zones.json, les autres caméras config/zones_<nom>.json."""
    def __init__(self, camera="webcam"):
        self.fichier = ZONES_FICHIER if camera in ("", "webcam") else REGLAGES / f"zones_{camera}.json"
        self.liste = json.loads(self.fichier.read_text()) if self.fichier.exists() else []

    def sauver(self):
        self.fichier.write_text(json.dumps(self.liste, ensure_ascii=False, indent=1))

    def ajouter(self, nom, points, W, H):
        self.liste = [z for z in self.liste if z["nom"] != nom]
        self.liste.append({"nom": nom, "points": [[round(x / W, 4), round(y / H, 4)] for x, y in points]})
        self.sauver()

    def retirer_derniere(self):
        z = self.liste.pop() if self.liste else None
        self.sauver()
        return z and z["nom"]

    def _poly(self, z, W, H):
        return (np.array(z["points"]) * [W, H]).astype(np.int32)

    def contenant(self, pt, W, H):
        return [z["nom"] for z in self.liste if cv2.pointPolygonTest(self._poly(z, W, H), pt, False) >= 0]

    def dessiner(self, img, alerte=()):
        H, W = img.shape[:2]
        for z in self.liste:
            p = self._poly(z, W, H)
            c = ROUGE if z["nom"] in alerte else ORANGE
            calque = img.copy()
            cv2.fillPoly(calque, [p], c)
            cv2.addWeighted(calque, 0.12, img, 0.88, 0, img)
            cv2.polylines(img, [p], True, c, 2, cv2.LINE_AA)
            texte(img, "ZONE " + z["nom"], tuple(p.min(axis=0) + [6, 20]), 0.5, c)


class Alertes:
    """Envoie les alertes sur Telegram. Sans bot (ou sans internet) elles attendent dans une file sur le disque
    et partent dès que c'est possible. Le bot répond aussi aux commandes /photo /armer /desarmer /etat /events."""
    API = "https://api.telegram.org/bot{}/{}"

    def __init__(self):
        self.fichier_attente = EVENEMENTS / "alertes_en_attente.jsonl"
        self.en_attente = [json.loads(l) for l in self.fichier_attente.read_text().splitlines()
                           if l.strip()] if self.fichier_attente.exists() else []
        self.charger_reglages()
        self.q, self.commandes, self.statut = queue.Queue(), {}, "pas de bot Telegram"
        threading.Thread(target=self._envoi, daemon=True).start()
        threading.Thread(target=self._ecoute, daemon=True).start()

    def charger_reglages(self):
        cfg = json.loads(TELEGRAM_FICHIER.read_text()) if TELEGRAM_FICHIER.exists() else {}
        self.token, self.chat_id = (cfg.get("token") or "").strip(), cfg.get("chat_id")

    def pret(self):
        return bool(self.token and self.chat_id)

    def envoyer(self, texte, photo=None, video=None):
        """photo / video = chemins de fichiers. Ne bloque jamais la vision."""
        self.q.put({"t": time.time(), "texte": texte, "photo": photo and str(photo), "video": video and str(video)})

    def _appel(self, methode, data=None, fichiers=None, timeout=60):
        r = requests.post(self.API.format(self.token, methode), data=data, files=fichiers, timeout=timeout)
        r.raise_for_status()
        return r.json()

    def _envoyer_un(self, a):
        d = {"chat_id": self.chat_id}
        legende = f"{a['texte']}\n{datetime.fromtimestamp(a['t']):%d/%m/%Y %H:%M:%S}"
        if a.get("video") and Path(a["video"]).exists():
            with open(a["video"], "rb") as f:
                self._appel("sendVideo", {**d, "caption": legende, "supports_streaming": True}, {"video": f}, 300)
        elif a.get("photo") and Path(a["photo"]).exists():
            with open(a["photo"], "rb") as f:
                self._appel("sendPhoto", {**d, "caption": legende}, {"photo": f})
        else:
            self._appel("sendMessage", {**d, "text": legende})

    def _sauver_attente(self):
        self.fichier_attente.write_text("".join(json.dumps(a, ensure_ascii=False) + "\n" for a in self.en_attente))

    def _envoi(self):
        while True:
            try:
                self.en_attente.append(self.q.get(timeout=20))
                self._sauver_attente()
            except queue.Empty:
                pass
            if not self.pret():
                self.charger_reglages()        # le jeton a peut-être été ajouté entre-temps
                continue
            while self.en_attente:
                try:
                    self._envoyer_un(self.en_attente[0])
                except Exception as e:
                    self.statut = f"envoi impossible ({type(e).__name__}), {len(self.en_attente)} en attente"
                    break
                self.en_attente.pop(0)
                self._sauver_attente()
                self.statut = "Telegram OK"

    def _ecoute(self):
        """Commandes envoyées au bot. Le PREMIER chat qui écrit au bot en devient le propriétaire."""
        offset = None
        while True:
            if not self.token:
                time.sleep(20); self.charger_reglages(); continue
            try:
                r = self._appel("getUpdates", {"timeout": 25, "offset": offset}, timeout=40)
            except Exception:
                time.sleep(15); continue
            for u in r.get("result", []):
                offset = u["update_id"] + 1
                msg = u.get("message") or {}
                chat, txt = msg.get("chat", {}).get("id"), (msg.get("text") or "").strip().lower()
                if chat is None:
                    continue
                if self.chat_id is None:
                    self.chat_id = chat
                    TELEGRAM_FICHIER.write_text(json.dumps({"token": self.token, "chat_id": chat}, indent=1))
                    self._repondre("✅ Vision Live est relié à ce chat. Commandes : /photo /armer /desarmer /etat /events")
                    continue
                if chat != self.chat_id:               # quelqu'un d'autre a trouvé le bot : on l'ignore
                    continue
                f = self.commandes.get(txt.split("@")[0].split()[0] if txt else "")
                try:
                    rep = f() if f else "Commandes : /photo /armer /desarmer /etat /events"
                except Exception as e:
                    rep = f"Erreur : {e}"
                if isinstance(rep, (bytes, np.ndarray)):
                    self._appel("sendPhoto", {"chat_id": self.chat_id}, {"photo": ("photo.jpg", bytes(rep))})
                else:
                    self._repondre(rep)

    def _repondre(self, s):
        try:
            self._appel("sendMessage", {"chat_id": self.chat_id, "text": s})
        except Exception:
            pass


class Enregistreur:
    """Garde en permanence les 10 dernières secondes (images réduites, en JPEG) dans un thread à part.
    Pendant un événement : continue d'enregistrer, cherche le meilleur visage, puis écrit le clip H.264."""
    def __init__(self, alertes):
        self.alertes = alertes
        self.q = queue.Queue(maxsize=8)
        self.tampon = deque(maxlen=CONFIG["surv_avant_s"] * CONFIG["surv_fps"])
        self.evt, self.lock, self.t_pousse = None, threading.Lock(), 0.0
        self.visages = cv2.FaceDetectorYN.create(str(MODELES / "visages" / "face_detection_yunet_2023mar.onnx"),
                                                 "", (320, 320), CONFIG["visage_seuil"])
        self.thread = threading.Thread(target=self._boucle, daemon=True)
        self.actif = True
        self.thread.start()

    def pousser(self, frame, dets, t):
        if t - self.t_pousse < 1 / CONFIG["surv_fps"]:
            return
        self.t_pousse = t
        boites = [(d["etiquette"], d["b"], d["nom"] == "personne") for d in dets]
        try:
            self.q.put_nowait((t, frame, boites))
        except queue.Full:                         # le thread est en retard : on saute une image plutôt que ralentir
            pass

    def demarrer(self, evt):
        with self.lock:
            if self.evt:                            # déjà en train d'enregistrer : on prolonge le même clip
                self.evt["fin"] = evt["t"] + CONFIG["surv_apres_s"]
                self.evt["messages"].append(evt["message"])
                return self.evt
            evt.update(fin=evt["t"] + CONFIG["surv_apres_s"], images=list(self.tampon), messages=[evt["message"]],
                       visage=None, visage_score=0.0)
            self.evt = evt
            return evt

    def _visage(self, frame, boites):
        """Cherche le visage le plus net dans le haut de chaque personne."""
        meilleur = None
        H, W = frame.shape[:2]
        for _, (x1, y1, x2, y2), personne in boites:
            if not personne:
                continue
            y2b = y1 + max(40, int((y2 - y1) * 0.6))          # le visage est dans le haut du corps
            crop = frame[y1:min(y2b, H), x1:x2]
            if crop.shape[0] < 40 or crop.shape[1] < 40:
                continue
            self.visages.setInputSize((crop.shape[1], crop.shape[0]))
            _, f = self.visages.detect(crop)
            for v in (f if f is not None else []):
                x, y, w, h, score = int(v[0]), int(v[1]), int(v[2]), int(v[3]), float(v[14])
                note = score * np.sqrt(w * h)                  # sûr ET grand = net
                if meilleur is None or note > meilleur[0]:
                    m = int(0.4 * max(w, h))
                    gx, gy = x1 + x, y1 + y
                    meilleur = (note, frame[max(0, gy - m):gy + h + m, max(0, gx - m):gx + w + m].copy())
        return meilleur

    def _boucle(self):
        L = CONFIG["surv_largeur"]
        while self.actif or not self.q.empty():
            try:
                t, frame, boites = self.q.get(timeout=0.5)
            except queue.Empty:
                self._peut_etre_finir(time.time())
                continue
            H, W = frame.shape[:2]
            k = L / W
            petit = cv2.resize(frame, (L, int(H * k)))
            for nom, (x1, y1, x2, y2), personne in boites:
                c = CYAN if personne else ORANGE
                cv2.rectangle(petit, (int(x1 * k), int(y1 * k)), (int(x2 * k), int(y2 * k)), c, 2)
                texte(petit, nom, (int(x1 * k) + 4, int(y1 * k) + 18), 0.5, c)
            texte(petit, datetime.fromtimestamp(t).strftime("%d/%m/%Y %H:%M:%S"), (10, 24), 0.6, BLANC)
            jpg = cv2.imencode(".jpg", petit, [cv2.IMWRITE_JPEG_QUALITY, 80])[1]
            self.tampon.append((t, jpg))
            with self.lock:
                evt = self.evt
            if evt:
                evt["images"].append((t, jpg))
                v = self._visage(frame, boites)
                if v and v[0] > evt["visage_score"]:
                    evt["visage_score"], evt["visage"] = v
            self._peut_etre_finir(t)

    def _peut_etre_finir(self, t):
        with self.lock:
            evt = self.evt
            if not evt or t < evt["fin"]:
                return
            self.evt = None
        self._ecrire(evt)

    def _ecrire(self, evt):
        d = evt["dossier"]
        clip = d / "clip.mp4"
        imgs = evt["images"]
        if imgs:
            premiere = cv2.imdecode(imgs[0][1], cv2.IMREAD_COLOR)
            h, w = premiere.shape[:2]
            with av.open(str(clip), "w") as sortie:          # H.264 : se lit directement sur un téléphone
                flux = sortie.add_stream("libx264", rate=CONFIG["surv_fps"])
                flux.width, flux.height, flux.pix_fmt = w, h, "yuv420p"
                flux.options = {"crf": "26", "preset": "veryfast"}
                for _, jpg in imgs:
                    img = cv2.imdecode(jpg, cv2.IMREAD_COLOR)
                    for paquet in flux.encode(av.VideoFrame.from_ndarray(img, format="bgr24")):
                        sortie.mux(paquet)
                for paquet in flux.encode():
                    sortie.mux(paquet)
        visage = None
        if evt["visage"] is not None and evt["visage"].size:
            visage = d / "visage.jpg"
            cv2.imwrite(str(visage), evt["visage"])
        info = {k: evt[k] for k in ("regle", "message", "messages", "zone", "objets")}
        info.update(debut=datetime.fromtimestamp(evt["t"]).isoformat(timespec="seconds"),
                    duree_clip_s=round(len(imgs) / CONFIG["surv_fps"], 1), visage=bool(visage))
        (d / "info.json").write_text(json.dumps(info, ensure_ascii=False, indent=1))
        resume = " / ".join(dict.fromkeys(evt["messages"]))
        if visage:
            self.alertes.envoyer(f"🙂 Visage vu pendant : {resume}", photo=visage)
        if imgs:
            self.alertes.envoyer(f"🎥 Clip : {resume}", video=clip)
        evt["fini"] = True

    def arreter(self):
        """Fin de session : termine proprement le clip en cours."""
        with self.lock:
            if self.evt:
                self.evt["fin"] = 0
        self.actif = False
        self.thread.join(timeout=30)
        self._peut_etre_finir(time.time())


class Surveillance:
    """Applique les RÈGLES aux objets fiables des images d'UNE caméra et déclenche les événements."""
    def __init__(self, zones, alertes, camera=""):
        self.zones, self.alertes, self.camera = zones, alertes, camera
        self.enreg = Enregistreur(alertes)
        self.arme, self.t_effet = False, 0.0
        self.etats, self.derniere_alerte = {}, {}
        self.zones_alerte, self.evenements = set(), []
        self.derniere = None
        alertes.commandes.update({
            "/photo": self._photo, "/armer": lambda: self.basculer(True), "/desarmer": lambda: self.basculer(False),
            "/etat": lambda: self.statut()[0], "/events": self._liste, "/start": lambda: self.statut()[0]})

    def basculer(self, armer=None):
        self.arme = (not self.arme) if armer is None else armer
        self.etats.clear()
        if self.arme:
            self.t_effet = time.time() + CONFIG["surv_delai"]
            return f"Surveillance armée dans {CONFIG['surv_delai']} secondes."
        return "Surveillance désarmée."

    def actif(self, t):
        if self.arme and t >= self.t_effet:
            return True
        return bool(CONFIG["surv_auto"]) and dans_horaires(CONFIG["surv_auto"])

    def statut(self):
        t = time.time()
        if self.arme and t < self.t_effet:
            return f"ARMEMENT dans {self.t_effet - t:.0f} s", ORANGE
        if self.actif(t):
            return f"ARMEE  |  {len(self.evenements)} alertes  |  {self.alertes.statut}", ROUGE
        return f"desarmee [G]  |  {len(self.alertes.en_attente)} alertes en attente", GRIS

    def _photo(self):
        if self.derniere is None:
            return "Pas encore d'image."
        return cv2.imencode(".jpg", self.derniere, [cv2.IMWRITE_JPEG_QUALITY, 85])[1].tobytes()

    def _liste(self):
        if not self.evenements:
            return "Aucune alerte depuis le lancement."
        return "\n".join(f"{datetime.fromtimestamp(e['t']):%H:%M:%S} — {e['message']}" for e in self.evenements[-10:])

    def _zones_de(self, regle, d, W, H):
        """Zones de la règle où se trouve l'objet. « * » = les zones dessinées, ou toute l'image s'il n'y en a aucune."""
        if regle["zones"] == "*" and not self.zones.liste:
            return ["image"]
        x1, y1, x2, y2 = d["b"]
        dedans = self.zones.contenant(((x1 + x2) / 2, (y1 + y2) / 2), W, H)
        return dedans if regle["zones"] == "*" else [z for z in dedans if z in regle["zones"]]

    def maj(self, frame, dets, t=None):
        """À appeler à chaque image analysée. Renvoie les messages d'alerte (à dire / afficher)."""
        t = time.time() if t is None else t
        self.derniere = frame
        self.enreg.pousser(frame, dets, t)
        self.zones_alerte = set()
        if not self.actif(t):
            return []
        H, W = frame.shape[:2]
        surs = [d for d in dets if not d["incertain"]]
        maintenant = datetime.fromtimestamp(t)
        messages = []
        for r in REGLES:
            if not dans_horaires(r.get("horaires"), maintenant):
                continue
            vus = {}                                    # clé → détection
            for d in surs:
                if d["nom"] in r["objets"]:
                    for z in self._zones_de(r, d, W, H):
                        vus[(r["nom"], z, d["nom"] if r["type"] == "disparition" else "")] = d
            for cle, d in vus.items():
                e = self.etats.setdefault(cle, {"depuis": t, "installe": False})
                e["vu"], e["boite"] = t, d["b"]
                if r["type"] == "presence":
                    self.zones_alerte.add(cle[1])
                    if t - e["depuis"] >= r["duree"]:
                        msg = f"Présence : {d['nom']}" + ("" if cle[1] == "image" else f" dans la zone {cle[1]}")
                        messages += self._declencher(r, cle, msg, frame, dets, t)
                elif t - e["depuis"] >= r["present"]:
                    e["installe"] = True
            for cle in [c for c in self.etats if c[0] == r["nom"] and c not in vus]:
                e = self.etats[cle]
                if r["type"] == "disparition" and e["installe"]:
                    if t - e["vu"] >= r["absent"]:
                        lieu = "" if cle[1] == "image" else f" de la zone {cle[1]}"
                        qui = " (une personne est là)" if any(d["nom"] == "personne" for d in surs) else ""
                        messages += self._declencher(r, cle, f"{cle[2].capitalize()} a disparu{lieu}{qui}", frame, dets, t)
                        del self.etats[cle]
                elif t - e["vu"] > 1.0:                 # disparu plus d'1 s : on repart de zéro
                    del self.etats[cle]
        return messages

    def _declencher(self, r, cle, message, frame, dets, t):
        if t - self.derniere_alerte.get(cle, -1e9) < r["pause"]:
            return []
        self.derniere_alerte[cle] = t
        if self.camera:                                 # « Présence : personne (iphone) »
            message = f"{message} ({self.camera})"
        nom = sans_accents(f"{self.camera}_{r['nom']}" if self.camera else r["nom"]).replace(" ", "_")
        dossier = EVENEMENTS / f"{datetime.fromtimestamp(t):%Y%m%d_%H%M%S}_{nom}"
        dossier.mkdir(parents=True, exist_ok=True)
        vue = frame.copy()
        for d in dets:
            x1, y1, x2, y2 = d["b"]
            c = ROUGE if d["nom"] in r["objets"] else CYAN
            cv2.rectangle(vue, (x1, y1), (x2, y2), c, 3)
            texte(vue, d["etiquette"], (x1 + 4, y1 + 22), 0.7, c, 2)
        self.zones.dessiner(vue, {cle[1]})
        cv2.imwrite(str(dossier / "image.jpg"), vue, [cv2.IMWRITE_JPEG_QUALITY, 88])
        evt = {"t": t, "regle": r["nom"], "zone": cle[1], "message": message, "dossier": dossier,
               "objets": sorted({d["nom"] for d in dets})}
        self.evenements.append(evt)
        self.enreg.demarrer(evt)
        self.alertes.envoyer(f"🚨 {message}", photo=dossier / "image.jpg")   # tout de suite, sans attendre le clip
        return [f"Alerte : {message}."]

    def arreter(self):
        self.enreg.arreter()


print(f"Surveillance prête — {len(REGLES)} règles, {len(Zones().liste)} zones sur la webcam, "
      f"Telegram {'configuré' if TELEGRAM_FICHIER.exists() else 'pas encore configuré (alertes gardées en attente)'}")

In [ ]:
# 8) VOIX : tu parles (touche V), il comprend (Whisper, en local) et il répond à voix haute
# Voix de réponse : Kokoro (la plus naturelle) ou Piper (instantanée), en local sur le processeur — voix SIWIS, licence
# CC-BY 4.0 (à citer dans le produit). Mesuré : Kokoro commence à parler en ~1,3 s, Piper en ~0,15 s, sans silence
# entre les phrases. Au démarrage, Piper (prêt en ~3 s) parle pendant que Kokoro se charge (~6-15 s).
# L'ancienne voix robot (espeak) ne sert plus que si aucune des deux ne marche.
import gc
import signal
import soundfile as sf
from scipy.signal import butter, sosfilt

# Un seul chargement de modèle à la fois (Whisper, Kokoro, Qwen) : pendant un chargement, transformers change un réglage
# commun à tout le programme, et deux chargements en même temps se marchent dessus (mesuré : Qwen cassé 2 fois sur 3).
CHARGEMENT = threading.Lock()

CONFIG.update({
    "voix_moteur": "kokoro",      # "kokoro" (naturelle) | "piper" (instantanée) | "espeak" (l'ancienne, robotique)
    "voix_vitesse": 1.0,          # vitesse de parole de Kokoro (1.0 = normale)
})

def parler_espeak(s):
    """L'ancienne voix (speech-dispatcher / espeak), dernier secours. Coupe la phrase précédente."""
    subprocess.run(["spd-say", "-C"], stderr=subprocess.DEVNULL)
    subprocess.Popen(["spd-say", "-l", "fr", "-r", "10", s], stderr=subprocess.DEVNULL)


class Parole:
    """Voix de réponse naturelle, en local sur le processeur (la carte graphique reste pour la vision).
    Phrase par phrase : la 1re est dite pendant que la suivante se prépare. Une nouvelle réponse coupe l'ancienne.
    Piper se charge d'abord (~3 s) et parle en attendant Kokoro ; ensuite c'est la voix choisie qui parle."""
    def __init__(self):
        self.q, self.voix, self.erreurs, self.proc = queue.Queue(), {}, {}, None
        self.a_charger = ["piper"] + ([CONFIG["voix_moteur"]] if CONFIG["voix_moteur"] != "piper" else [])
        self.numero, self.n_fichier = 0, 0
        threading.Thread(target=self._charger_tout, daemon=True).start()
        threading.Thread(target=self._boucle, daemon=True).start()

    @property
    def etat(self):
        if self.voix:
            return "pret"
        return "erreur" if len(self.erreurs) == len(self.a_charger) else "chargement"

    def dire(self, texte):
        self.numero += 1                                 # tout ce qui était en cours devient périmé
        if self.proc is not None and self.proc.poll() is None:
            self.proc.terminate()
        self.q.put((self.numero, texte))

    def _charger(self, moteur):
        d = MODELES / "voix"
        if moteur == "kokoro":
            from kokoro import KModel, KPipeline
            k = d / "kokoro"
            modele = KModel(config=str(k / "config.json"), model=str(k / "kokoro-v1_0.pth")).eval()
            pipe = KPipeline(lang_code="f", model=modele, repo_id="hexgrad/Kokoro-82M")
            voix = str(k / "voices" / "ff_siwis.pt")
            def fabriquer(texte):                        # phrase par phrase : la 1re part au bout de ~1 s
                for phrase in re.split(r"(?<=[.!?…])\s+", texte):
                    for r in pipe(phrase, voice=voix, speed=CONFIG["voix_vitesse"]):
                        if r.audio is not None:
                            yield r.audio.numpy(), 24000
            return fabriquer
        if moteur == "piper":
            from piper import PiperVoice
            v = PiperVoice.load(str(d / "piper" / "fr_FR-siwis-medium.onnx"))
            return lambda t: ((np.frombuffer(c.audio_int16_bytes, np.int16).astype(np.float32) / 32767, c.sample_rate)
                              for c in v.synthesize(t))
        raise ValueError(f"voix inconnue : {moteur}")

    def _charger_tout(self):
        for moteur in self.a_charger:
            t = time.time()
            try:
                with CHARGEMENT:
                    fabriquer = self._charger(moteur)
                list(fabriquer("Bonjour."))              # préchauffage : la 1re vraie phrase partira plus vite
                self.voix[moteur] = fabriquer
                print(f"Voix {moteur} prête en {time.time() - t:.1f} s", flush=True)
            except Exception as e:
                self.erreurs[moteur] = f"{type(e).__name__}: {str(e)[:120]}"
                print(f"Voix {moteur} indisponible : {self.erreurs[moteur]}", flush=True)

    def _meilleure(self):
        """La voix choisie si elle est prête, sinon Piper (pendant le chargement de Kokoro)."""
        return self.voix.get(CONFIG["voix_moteur"]) or next(iter(self.voix.values()), None)

    def _jouer(self, audio, sr):
        """Écrit le morceau et le joue dès que le précédent est fini (2 fichiers en alternance)."""
        self.n_fichier ^= 1
        wav = Path(f"/tmp/vision_live_parole_{self.n_fichier}.wav")
        sf.write(wav, audio, sr)
        if self.proc is not None:
            self.proc.wait()
        self.proc = subprocess.Popen(["paplay", str(wav)], stderr=subprocess.DEVNULL)

    def _boucle(self):
        while True:
            numero, texte = self.q.get()
            t = time.time()
            while self.etat == "chargement" and time.time() - t < 15:   # les toutes premières secondes : on attend Piper
                time.sleep(0.1)
            if numero != self.numero:                    # déjà remplacé par une réponse plus récente
                continue
            fabriquer = self._meilleure()
            if fabriquer is None:                        # aucune voix naturelle ne marche
                parler_espeak(texte)
                continue
            try:
                for audio, sr in fabriquer(texte):
                    if numero != self.numero:
                        break
                    self._jouer(audio, sr)
            except Exception as e:
                print(f"Voix : erreur {type(e).__name__}: {str(e)[:120]}", flush=True)
                parler_espeak(texte)


def parler(s):
    """Dit une réponse à voix haute (Kokoro, Piper pendant son chargement, espeak seulement en dernier secours)."""
    if CONFIG["voix_moteur"] == "espeak" or PAROLE is None:
        return parler_espeak(s)
    PAROLE.dire(s)

PAROLE = Parole() if CONFIG["voix_moteur"] != "espeak" else None   # se charge en arrière-plan dès maintenant


def repetitif(txt):
    """Whisper 'hallucine' parfois sur du bruit : il répète la même phrase en boucle."""
    mots = txt.lower().split()
    return len(mots) >= 6 and len(set(mots)) / len(mots) < 0.4

class Voix:
    """Micro (arecord) → Whisper (transformers) → texte. Tout tourne en arrière-plan : la vidéo ne gèle jamais.
    Whisper est sur la carte graphique (0,25 s par phrase). Quand Qwen l'occupe, il passe sur le processeur
    (~4 s par phrase) pour lui laisser la place : sinon la carte déborde et tout s'arrête (vu le 30/10)."""
    FILTRE = butter(4, 120, "highpass", fs=16000, output="sos")    # enlève ronflements / bruit grave

    def __init__(self):
        self.asr, self.etat, self.erreur = None, "chargement", None
        self.sur_gpu = torch.cuda.is_available()
        self.lock = threading.Lock()                     # une transcription et un déménagement jamais en même temps
        self.resultats = queue.Queue()
        self.proc, self.t_debut = None, 0.0
        self.wav = Path("/tmp/vision_live_question.wav")
        threading.Thread(target=self._charger, daemon=True).start()

    def _charger(self):
        try:
            from transformers import pipeline
            with self.lock, CHARGEMENT:
                self.asr = pipeline("automatic-speech-recognition", model=CONFIG["whisper"],
                                    device=0 if self.sur_gpu else -1,
                                    dtype=torch.float16 if self.sur_gpu else torch.float32)
            if self.etat == "chargement":
                self.etat = "pret"
        except Exception as e:
            self.etat, self.erreur = "erreur", str(e)[:200]

    def deplacer(self, gpu):
        """Met Whisper sur la carte graphique (gpu=True) ou sur le processeur (gpu=False) — appelé par Qwen (8b).
        La place sur la carte est libérée tout de suite ; le rechargement se fait en arrière-plan (~3 s)."""
        gpu = gpu and torch.cuda.is_available()
        if gpu == self.sur_gpu:
            return
        with self.lock:                                  # attend la fin d'une transcription en cours
            self.sur_gpu, self.asr = gpu, None
            if self.etat == "pret":
                self.etat = "chargement"
            gc.collect()
            torch.cuda.empty_cache()
        threading.Thread(target=self._charger, daemon=True).start()

    def demarrer(self):
        if self.etat != "pret" or self.proc:
            return False
        self.proc = subprocess.Popen(["arecord", "-q", "-f", "S16_LE", "-r", "16000", "-c", "1", "-t", "wav",
                                      "-d", str(CONFIG["voix_max_s"]), str(self.wav)])
        self.t_debut, self.etat = time.time(), "ecoute"
        return True

    def arreter(self):
        if self.proc:
            if self.proc.poll() is None:
                self.proc.send_signal(signal.SIGINT)   # arecord finalise proprement le fichier
            self.proc.wait(); self.proc = None
            self.etat = "reflexion"
            threading.Thread(target=self._transcrire, daemon=True).start()

    def maj(self):
        """À appeler à chaque image : arrêt auto quand la durée max est atteinte."""
        if self.proc and self.proc.poll() is not None:
            self.arreter()

    def _transcrire(self):
        try:
            a, sr = sf.read(self.wav, dtype="float32")
            a = a - a.mean()
            if len(a) < sr * 0.4:
                self.resultats.put(""); return
            if (np.abs(a) > 0.99).mean() > 0.05:      # plus de 5 % du son écrêté → micro trop fort
                self.resultats.put("__erreur__ micro saturé : baisse le volume d'entrée (Paramètres > Son > Entrée)")
                return
            a = sosfilt(self.FILTRE, a).astype("float32")
            if np.sqrt((a ** 2).mean()) < 0.002:
                self.resultats.put(""); return
            a = a / max(np.abs(a).max(), 1e-3) * 0.9            # normalise le volume
            t = time.time()
            while self.asr is None and time.time() - t < 20:    # Whisper est en train de changer de place
                time.sleep(0.1)
            with self.lock:
                txt = self.asr({"raw": a, "sampling_rate": sr},
                               generate_kwargs={"language": "french", "task": "transcribe",
                                                "max_new_tokens": 40, "no_repeat_ngram_size": 3})["text"].strip()
            self.resultats.put("" if repetitif(txt) or set(txt) <= set(". ") else txt)
        except Exception as e:
            self.resultats.put(f"__erreur__ {e}")
        finally:
            self.etat = "pret" if self.asr is not None else "chargement"


# ---------- Compréhension des phrases (français) ----------
def _norm(s):
    s = sans_accents(s.lower()).replace("'", " ").replace("-", " ")
    return re.sub(r"[^a-z0-9 ]", " ", s).split()

DETERMINANTS = set("mon ma mes le la les un une des du de d l ton ta tes son sa ses notre votre leur".split())

INTENTIONS = [   # (action, déclencheurs) — testés dans l'ordre
    ("corriger", ["non c est", "en fait c est", "ce n est pas", "c est pas", "c est un", "c est une", "ceci est"]),
    ("oublier",  ["oublie"]),
    ("ou",       ["ou est", "ou sont", "ou j ai mis", "ou ai je mis", "tu as vu", "as tu vu", "t as vu", "ou se trouve"]),
    ("chercher", ["trouve", "cherche", "montre moi", "detecte", "localise", "repere"]),
    ("combien",  ["combien"]),
    ("decrire",  ["qu est ce que tu vois", "que vois tu", "tu vois quoi", "decris", "qu est ce qu il y a", "dis moi ce que"]),
    ("stop",     ["arrete", "stop", "annule", "laisse tomber"]),
    ("mission",  ["mission", "on joue", "jouer"]),
    ("hd",       ["haute precision", "precision"]),
    ("mode",     ["mode", "vision"]),
]
MODES_VOIX = {"nuit": 4, "nocturne": 4, "spotlight": 2, "projecteur": 2, "chaleur": 3, "heatmap": 3,
              "thermique": 3, "normal": 1, "normale": 1, "hud": 1}
PAS_UN_OBJET = ["pas un objet", "rien du tout", "c est rien", "ma main", "une main", "mon visage", "du flou"]

def trouver_objet(mots, connus):
    """Cherche dans la phrase le nom d'objet connu le plus long (tolère les petites fautes de Whisper)."""
    phrase = " ".join(m for m in mots if m not in DETERMINANTS)
    index = {" ".join(_norm(n)): n for n in connus}
    for cle in sorted(index, key=len, reverse=True):
        if re.search(rf"\b{re.escape(cle)}s?\b", phrase):
            return index[cle], phrase
    proches = difflib.get_close_matches(phrase, list(index), n=1, cutoff=0.75)
    if not proches:   # essaie chaque groupe de 1 à 3 mots
        mm = phrase.split()
        groupes = [" ".join(mm[i:i + k]) for k in (3, 2, 1) for i in range(len(mm) - k + 1)]
        for g in groupes:
            proches = difflib.get_close_matches(g, list(index), n=1, cutoff=0.8)
            if proches: break
    return (index[proches[0]], phrase) if proches else (None, phrase)

def comprendre(texte, connus):
    """Retourne (action, objet, reste_de_phrase). Pour une correction, objet = "REJET" si « ce n'est pas un objet »."""
    mots = _norm(texte)
    phrase = " ".join(mots)
    for action, declencheurs in INTENTIONS:
        for d in declencheurs:
            if re.search(rf"\b{d}\b", phrase):
                reste = phrase.split(d, 1)[1]
                if action == "corriger":
                    if any(p in phrase for p in PAS_UN_OBJET):
                        return "corriger", "REJET", ""
                    if " c est " in f" {reste} ":                 # « ce n'est pas un X, c'est un Y » → Y
                        reste = f" {reste} ".rsplit(" c est ", 1)[1]
                if action == "mode":
                    m = next((MODES_VOIX[w] for w in mots if w in MODES_VOIX), None)
                    return ("mode", m, "") if m else ("inconnu", None, phrase)
                obj, libre = trouver_objet(reste.split(), connus)
                return action, obj, libre
    obj, libre = trouver_objet(mots, connus)
    return ("ou", obj, libre) if obj else ("inconnu", None, phrase)

# Petit test hors-ligne de la compréhension
_connus = list(VOCAB.values())
for q in ["Où est mon téléphone ?", "trouve ma clé USB", "Qu'est-ce que tu vois ?", "passe en mode nuit",
          "combien de bouteilles", "tu as vu mes lunettes", "cherche une bouteille bleue",
          "Non, c'est un cordon badge", "ce n'est pas un téléphone, c'est un routeur", "ce n'est pas un objet, c'est ma main"]:
    print(f"{q:45s} → {comprendre(q, _connus)}")
print("Anti-boucle :", repetitif("c'est bon, " * 20), "/", repetitif("où est mon téléphone"))
print(f"Voix de réponse : {CONFIG['voix_moteur']} (se charge en arrière-plan, Piper parle en attendant)")

In [ ]:
# 8b) NIVEAU 4 — EXPERT : Qwen3-VL-4B, un modèle qui COMPREND l'image et répond en français (à la demande)
# I = « que se passe-t-il ? » sur l'image en direct. Les questions que le système ne sait pas traiter seul
# (« que fait la personne ? », « explique... », « de quelle couleur est... ») lui sont envoyées, à l'écrit (F) comme à la voix (V).
# Il décrit aussi chaque alerte de surveillance (description.txt + message Telegram ; commande /scene sur Telegram).
# MÉMOIRE GRAPHIQUE (5,6 Go utilisables) : Qwen en prend 4,3. Pour ne plus jamais déborder (le 30/10 : V + Qwen + 2 caméras
# → la fenêtre s'est fermée) : quand Qwen se réveille, Whisper passe sur le processeur ; Qwen se rendort tout seul
# après `expert_veille_s` sans question (sauf surveillance armée) et Whisper revient sur la carte graphique.
# Mesuré (scénario du plantage rejoué) : plus de fermeture, pic 5,15 Go au lieu de 5,6.
import gc
import sys
sys.path.insert(0, str(MODELES))
from charger_qwen import QWEN, charger as charger_qwen, demander as demander_qwen, decharger as decharger_qwen

CONFIG.update({
    "expert": True,
    "expert_alertes": True,       # Qwen décrit chaque alerte de surveillance
    "expert_cote": 640,           # taille de l'image donnée à Qwen (mesuré : à 320 px il invente des détails)
    "expert_mots": 120,           # longueur max d'une réponse (en tokens)
    "expert_veille_s": 120,       # Qwen libère la carte graphique après 2 min sans question
})

QUESTION_SCENE = ("Décris en deux phrases courtes ce qui se passe sur cette image : les personnes, ce qu'elles font "
                  "et les objets importants. Réponds en français.")
QUESTION_ALERTE = ("Image d'une caméra de surveillance. En une phrase en français, décris la personne ou l'animal "
                   "visible (apparence, vêtements) et ce qu'il fait ou tient en main.")
CONSIGNE = " Réponds en français, en une ou deux phrases courtes."
# Questions qui demandent de COMPRENDRE la scène (et pas seulement de lister les objets) → Qwen
POUR_EXPERT = ["qu est ce qui se passe", "que se passe", "explique", "analyse", "que fait", "qu est ce qu il fait",
               "qu est ce qu elle fait", "en detail", "raconte", "lis ", "quelle couleur", "de quelle couleur", "pourquoi",
               "qui est", "a quoi ressemble", "est ce que quelqu un", "est ce que la ", "est ce que le ", "est ce que les "]

def pour_expert(texte):
    phrase = " ".join(_norm(texte)) + " "
    return any(d in phrase for d in POUR_EXPERT)


class Expert:
    """Qwen3-VL dans un thread à part : la vidéo ne gèle jamais pendant qu'il réfléchit (~6 s)."""
    def __init__(self):
        self.qwen, self.etat, self.erreur = None, "absent", None   # absent | chargement | pret | reflexion | erreur
        self.jobs, self.resultats = queue.Queue(), queue.Queue()
        self.n, self.duree, self.t_dernier = 0, 0.0, 0.0
        self.garder_eveille = False                      # True pendant que la surveillance est armée
        threading.Thread(target=self._prechauffer_disque, daemon=True).start()
        threading.Thread(target=self._boucle, daemon=True).start()

    @staticmethod
    def _prechauffer_disque():
        """Lit une fois le fichier de Qwen (4 Go) en tâche de fond, sans toucher à la carte graphique : il reste dans la
        mémoire du système, et le 1er réveil ne relit plus le disque (vu : ~20 s de réveil au lieu de ~5)."""
        try:
            with open(QWEN / "model.safetensors", "rb") as f:
                while f.read(16 << 20):
                    time.sleep(0.005)                    # doucement : le démarrage du reste passe avant
        except OSError:
            pass

    def demander(self, image, question=QUESTION_SCENE, rappel=None):
        """Renvoie tout de suite un message d'attente — et le DIT, pour qu'on sache que la demande est prise.
        La réponse arrive dans self.resultats (ou est passée à rappel)."""
        self.jobs.put((image.copy(), question, rappel))
        if rappel is not None:                           # demande interne (alerte, Telegram) : rien à dire
            return ""
        n = self.jobs.qsize()
        msg = ("Un instant, je réveille Qwen et je regarde." if self.qwen is None else
               "Je regarde." if n <= 1 else f"Je regarde ({n} questions en attente).")
        parler(msg)
        return msg

    def precharger(self):
        if self.qwen is None and self.etat != "chargement":
            self.jobs.put(None)

    def _reveiller(self):
        self.etat, t = "chargement", time.time()
        voix = globals().get("VOIX")
        if voix:
            voix.deplacer(gpu=False)                     # d'abord faire de la place sur la carte graphique
        try:
            with CHARGEMENT:                             # pas en même temps que Whisper ou Kokoro (cellule 8)
                self.qwen = self._reessayer(charger_qwen)
        except Exception:
            if voix:
                voix.deplacer(gpu=True)
            raise
        print(f"Qwen réveillé en {time.time() - t:.1f} s", flush=True)

    def _endormir(self):
        decharger_qwen(self.qwen)
        self.qwen, self.etat = None, "absent"
        voix = globals().get("VOIX")
        if voix:
            voix.deplacer(gpu=True)                      # Whisper retrouve la carte graphique (0,25 s par phrase)
        print("Qwen endormi : carte graphique libérée", flush=True)

    @staticmethod
    def _reessayer(fonction):
        """Si la carte graphique est pleine un instant : on libère le cache et on réessaie une fois."""
        try:
            return fonction()
        except torch.OutOfMemoryError:
            gc.collect()
            torch.cuda.empty_cache()
            return fonction()

    def _boucle(self):
        while True:
            try:
                job = self.jobs.get(timeout=5)
            except queue.Empty:                          # pas de question : faut-il libérer la carte graphique ?
                if (self.qwen is not None and not self.garder_eveille
                        and time.time() - self.t_dernier > CONFIG["expert_veille_s"]):
                    self._endormir()
                continue
            self.t_dernier = time.time()
            try:
                if self.qwen is None:
                    self._reveiller()
                self.etat = "pret"
                if job is None:                          # simple préchargement
                    continue
                img, question, rappel = job
                self.etat, t = "reflexion", time.time()
                rep = self._reessayer(lambda: demander_qwen(self.qwen, img, question, cote_max=CONFIG["expert_cote"],
                                                            max_new_tokens=CONFIG["expert_mots"]))
                self.n, self.duree, self.etat = self.n + 1, time.time() - t, "pret"
                self.t_dernier = time.time()
                (rappel or self.resultats.put)(rep)
            except Exception as e:
                self.etat, self.erreur = "erreur", f"{type(e).__name__}: {str(e)[:120]}"
                print(f"Qwen : {self.erreur}", flush=True)
                if job:
                    (job[2] or self.resultats.put)(f"Qwen n'a pas pu répondre ({self.erreur}).")


class SurveillanceExperte(Surveillance):
    """Surveillance (cellule 7c) + Qwen : chaque alerte reçoit une description en français."""
    def __init__(self, zones, alertes, camera=""):
        super().__init__(zones, alertes, camera)
        alertes.commandes["/scene"] = self._scene

    def basculer(self, armer=None):
        msg = super().basculer(armer)
        if CONFIG["expert_alertes"]:
            EXPERT.garder_eveille = self.arme            # armée : Qwen reste prêt pour décrire les alertes
            if self.arme:
                EXPERT.precharger()
        return msg

    def _declencher(self, r, cle, message, frame, dets, t):
        msgs = super()._declencher(r, cle, message, frame, dets, t)
        if msgs and CONFIG["expert_alertes"]:
            evt = self.evenements[-1]
            def decrire(rep):
                (evt["dossier"] / "description.txt").write_text(rep + "\n")
                self.alertes.envoyer(f"🧠 {evt['message']} — {rep}")
                EXPERT.resultats.put(rep)
            EXPERT.demander(frame, QUESTION_ALERTE, rappel=decrire)
        return msgs

    def _scene(self):
        """Commande Telegram /scene : Qwen décrit l'image en direct (attend sa réponse, 90 s max)."""
        if self.derniere is None:
            return "Pas encore d'image."
        fini, rep = threading.Event(), []
        EXPERT.demander(self.derniere, QUESTION_SCENE, rappel=lambda r: (rep.append(r), fini.set()))
        return rep[0] if fini.wait(90) else "Qwen met trop de temps à répondre."


EXPERT = Expert() if CONFIG["expert"] else None
for q in ["Que fait la personne ?", "Qu'est-ce que tu vois ?", "De quelle couleur est sa chemise ?", "Où est mon téléphone ?"]:
    print(f"{q:40s} → {'Qwen' if pour_expert(q) else 'réponse rapide (YOLOE)'}")
print(f"Expert {'prêt : Qwen se réveille au premier besoin, se rendort après ' + str(CONFIG['expert_veille_s']) + ' s' if EXPERT else 'désactivé'} ({QWEN})")

In [ ]:
# 9) Panneau latéral (tableau de bord en direct)
def couper(s, n):
    """Coupe un texte en lignes de n caractères max."""
    mots, lignes, l = sans_accents(s).split(), [], ""
    for m in mots:
        if len(l) + len(m) + 1 > n:
            lignes.append(l); l = m
        else:
            l = f"{l} {m}".strip()
    return lignes + ([l] if l else [])

def titre(p, s, y, c=CYAN, droite=None):
    w = p.shape[1]
    texte(p, s, (16, y), 0.55, c)
    if droite:
        texte(p, droite, (150, y), 0.42, BLANC)
    cv2.line(p, (16, y + 8), (w - 16, y + 8), (70, 70, 40), 1)
    return y + 28

def dessiner_panneau(p, etat, compte_live, journal, mission, voix):
    p[:] = (18, 14, 10)
    h, w = p.shape[:2]
    cv2.line(p, (0, 0), (0, h), CYAN, 2)
    texte(p, "VISION LIVE", (16, 36), 0.95, CYAN, 2)
    texte(p, f"YOLOE-11M + MobileCLIP2  |  {GPU_NOM}", (16, 58), 0.38, GRIS)
    if etat["rec"] and int(time.time() * 2) % 2:
        cv2.circle(p, (w - 30, 28), 8, ROUGE, -1)

    # --- état
    y = 84
    fps_c = VERT if etat["fps"] >= 20 else ORANGE if etat["fps"] >= 10 else ROUGE
    texte(p, f"FPS {etat['fps']:.0f}", (16, y), 0.48, fps_c)
    texte(p, f"{etat['ms']:.0f} ms", (100, y), 0.48, BLANC)
    texte(p, "HD" if etat["hd"] else "SD", (170, y), 0.48, OR if etat["hd"] else GRIS)
    texte(p, f"conf {CONFIG['conf']:.2f}", (215, y), 0.48, BLANC)
    texte(p, MODES[etat["mode"]], (305, y), 0.42, CYAN)
    y += 20
    texte(p, f"{len(cerveau.noms_fr)} objets connus  |  {len(cerveau.perso)} appris  |  {len(journal.vus)} vus", (16, y), 0.42, GRIS)
    if apprentissage:                                   # niveaux 2 et 3
        y += 18
        adapte = "adapte a ta piece" if apprentissage.W is not None else "pas encore entraine"
        texte(p, f"{len(apprentissage.ex)} corrections  |  {len(apprentissage.a_revoir)} a revoir [E]  |  {adapte}",
              (16, y), 0.42, OR if apprentissage.W is not None else GRIS)
    if "surv" in etat:                                  # surveillance (cellule 7c)
        y += 18
        s, c = etat["surv"]
        texte(p, f"SURVEILLANCE {s}"[:60], (16, y), 0.42, c)
    e = globals().get("EXPERT")
    if e:                                               # niveau 4 (cellule 8b)
        y += 18
        s = {"absent": "en veille  [I] = que se passe-t-il ?", "chargement": "se reveille...",
             "pret": f"pret [I]  |  {e.n} reponses ({e.duree:.0f} s la derniere)", "reflexion": "reflechit...",
             "erreur": f"erreur : {e.erreur}"}[e.etat]
        texte(p, f"QWEN {s}"[:60], (16, y), 0.42, ROUGE if e.etat == "erreur" else GRIS if e.etat == "absent" else OR)

    # --- en direct
    y = titre(p, "EN DIRECT", y + 26)
    if not compte_live:
        texte(p, "rien detecte...", (16, y), 0.45, GRIS); y += 21
    for (nom, perso), n in compte_live.most_common(5):
        c = couleur(nom, perso)
        bar = int(min(n, 6) / 6 * (w - 230))
        cv2.rectangle(p, (190, y - 12), (190 + max(bar, 4), y - 2), c, -1)
        texte(p, ("* " if perso else "") + nom[:20], (16, y), 0.45, c)
        texte(p, str(n), (w - 35, y), 0.45, BLANC)
        y += 21

    # --- assistant (voix + mémoire)
    statut = {"chargement": ("chargement de Whisper...", GRIS), "pret": ("[V] pour parler", VERT),
              "ecoute": (f"J'ECOUTE... {CONFIG['voix_max_s'] - (time.time() - voix.t_debut):.0f}s", ROUGE),
              "reflexion": ("je reflechis...", ORANGE), "erreur": ("voix indisponible", ROUGE)}[voix.etat]
    y = titre(p, "ASSISTANT", y + 8, OR, statut[0])
    if voix.etat == "ecoute":   # onde animée
        for i in range(30):
            a = int(10 * abs(np.sin(time.time() * 9 + i * 0.7)) * (0.5 + random.random() / 2))
            cv2.line(p, (16 + i * 12, y - a), (16 + i * 12, y + a), ROUGE, 3)
        y += 22
    if etat["question"]:
        for l in couper("Toi : " + etat["question"], 44)[:2]:
            texte(p, l, (16, y), 0.42, BLANC); y += 18
    if etat["reponse"]:
        x_txt, img = 16, etat.get("vignette")
        if img is not None:     # photo de l'endroit où l'objet a été vu
            v = cv2.resize(img, (96, 72))
            p[y - 12:y + 60, 16:112] = v
            cv2.rectangle(p, (16, y - 12), (112, y + 60), OR, 1)
            x_txt = 122
        y0 = y
        for l in couper(etat["reponse"], 36 if img is not None else 46)[:5]:
            texte(p, l, (x_txt, y), 0.42, OR); y += 18
        y = max(y, y0 + 76) if img is not None else y

    # --- mission (compacte)
    if mission.cible:
        y = titre(p, "MISSION", y + 8, ORANGE, f"score {mission.score}")
        texte(p, f"Montre-moi : {mission.cible.upper()}", (16, y), 0.5, ORANGE); y += 14
        r = mission.restant()
        cv2.rectangle(p, (16, y), (w - 16, y + 6), (50, 50, 50), -1)
        cv2.rectangle(p, (16, y), (16 + int((w - 32) * r / mission.DUREE_MAX), y + 6), VERT if r > 10 else ROUGE, -1)
        if mission.progression() > 0:
            cv2.rectangle(p, (16, y + 9), (16 + int((w - 32) * mission.progression()), y + 13), OR, -1)
        y += 26

    # --- journal
    y = titre(p, "JOURNAL", y + 6)
    for s, c in journal.lignes:
        if y > h - 64:
            break
        texte(p, s[:46], (16, y), 0.4, c); y += 18

    texte(p, "[V] parler [F] ecrire [I] qwen [A] apprendre [M]", (16, h - 46), 0.36, GRIS)
    texte(p, "[C] corriger [E] reviser [U] entrainer [H] HD", (16, h - 28), 0.36, OR)
    texte(p, "[G] surv. [Z] zone [D] camera [1-4] [T] [S] [R] [Q]", (16, h - 10), 0.36, ROUGE)

print("Panneau prêt")

In [ ]:
# 10) Boucle principale — VISION LIVE v6 (plusieurs caméras + apprentissage + surveillance + Qwen)
def executer(action, obj, libre, ctx):
    """Exécute une demande (vocale ou écrite). Renvoie (réponse, vignette ou None)."""
    etat, memoire, mission, compte = ctx["etat"], ctx["memoire"], ctx["mission"], ctx["compte"]
    visibles = set(compte)
    if action == "ou":
        if not obj:
            return "Quel objet ? Dis par exemple : où est mon téléphone.", None
        e = memoire.d.get(obj)
        img = cv2.imread(e["img"]) if e and Path(e["img"]).exists() else None
        if obj in visibles:
            etat["cible"] = obj
        return memoire.repondre(obj, visibles), img
    if action == "chercher":
        if not obj:
            if not libre:
                return "Que dois-je chercher ?", None
            cerveau.chercher_texte(libre, libre)     # objet inconnu → on l'ajoute au vocabulaire en direct
            obj = libre
        etat["cible"], etat["cible_annoncee"] = obj, False
        return (f"{obj} : je le vois, je te le montre !" if obj in visibles
                else f"Je cherche {obj}. Montre-le à la caméra."), None
    if action == "combien":
        n = compte.get(obj, 0) if obj else sum(compte.values())
        return (f"Je vois {n} {obj}." if obj else f"Je vois {n} objets en tout."), None
    if action == "decrire":
        if not compte:
            return "Je ne vois rien de particulier pour le moment.", None
        return "Je vois : " + ", ".join(f"{n} {nom}" if n > 1 else nom for nom, n in compte.most_common(8)) + ".", None
    if action == "stop":
        etat["cible"], mission.cible = None, None
        return "D'accord, j'arrête.", None
    if action == "mission":
        return f"Mission : montre-moi {mission.nouvelle()} !", None
    if action == "hd":
        etat["hd"] = not etat["hd"]
        return f"Haute précision {'activée' if etat['hd'] else 'désactivée'}.", None
    if action == "mode":
        etat["mode"] = obj
        return f"Mode {MODES[obj].lower()}.", None
    if action == "oublier":
        if obj in cerveau.perso:
            cerveau.oublier(obj)
            return f"J'ai oublié {obj}.", None
        return "Je ne peux oublier que les objets que tu m'as appris.", None
    return "Je n'ai pas compris. Essaie : où est mon téléphone, trouve ma clé USB, ou qu'est-ce que tu vois.", None


def lancer():
    global VOIX, ALERTES
    if "VOIX" not in globals():
        VOIX = Voix()          # Whisper se charge en arrière-plan (une seule fois)
    if "ALERTES" not in globals():
        ALERTES = Alertes()    # les threads Telegram ne sont lancés qu'une fois, même si on relance la fenêtre
    voix = VOIX

    W, H = CONFIG["largeur"], CONFIG["hauteur"]   # toutes les caméras sont affichées à ce format (sans déformation)
    PW = CONFIG["panneau"]
    canvas = np.zeros((H, W + PW, 3), np.uint8)
    journal, memoire = Journal(), Memoire()
    mission = Mission(journal)

    # --- une Caméra + une Perception (son propre suivi) + ses Zones + sa Surveillance par source
    sources = [(n, s) for n, s in CONFIG["cameras"].items() if s is not None]
    if not sources:
        raise RuntimeError("Aucune caméra dans CONFIG['cameras'] (cellule 5)")
    plusieurs = len(sources) > 1
    Surv = SurveillanceExperte if EXPERT else Surveillance          # + description des alertes par Qwen
    cams = {}
    for i, (nom, source) in enumerate(sources):
        z = Zones(nom)
        cams[nom] = {"cam": Camera(nom, source), "perception": Perception(base=i * 10_000_000), "zones": z,
                     "surv": Surv(z, ALERTES, nom if plusieurs else ""), "heat": Heatmap(H, W),
                     "trainees": defaultdict(lambda: deque(maxlen=32)), "dets": [], "frame": None, "coupee": False}
    for C in cams.values():                       # la webcam met quelques secondes à régler son exposition
        C["cam"].attendre(8 if isinstance(C["cam"].source, int) else 5)
    if not any(C["cam"].en_ligne for C in cams.values()):
        for C in cams.values():
            C["cam"].stop()
        raise RuntimeError("Aucune caméra ne donne d'image (webcam branchée ? DroidCam ouvert sur l'iPhone ?)")
    noms = list(cams)
    actif = next(n for n in noms if cams[n]["cam"].en_ligne)       # la caméra affichée en grand (Tab pour changer)

    # --- Telegram : les commandes portent sur TOUTES les caméras
    def basculer_tout(armer=None):
        armer = (not cams[actif]["surv"].arme) if armer is None else armer
        return [C["surv"].basculer(armer) for C in cams.values()][0]

    def photo_toutes():
        imgs = [cv2.resize(C["frame"], (640, 360)) for C in cams.values() if C["frame"] is not None and not C["coupee"]]
        return cv2.imencode(".jpg", cv2.hconcat(imgs), [cv2.IMWRITE_JPEG_QUALITY, 85])[1].tobytes() if imgs \
            else "Pas encore d'image."

    def liste_toutes():
        evts = sorted((e for C in cams.values() for e in C["surv"].evenements), key=lambda e: e["t"])[-10:]
        return "\n".join(f"{datetime.fromtimestamp(e['t']):%H:%M:%S} — {e['message']}" for e in evts) \
            or "Aucune alerte depuis le lancement."

    def etat_toutes():
        return "\n".join(f"{n} : {'COUPÉE' if C['coupee'] else 'en ligne' if C['cam'].en_ligne else 'DÉCONNECTÉE'}"
                         f" | {C['surv'].statut()[0]}"
                         for n, C in cams.items())

    ALERTES.commandes.update({"/photo": photo_toutes, "/armer": lambda: basculer_tout(True),
                              "/desarmer": lambda: basculer_tout(False), "/etat": etat_toutes,
                              "/events": liste_toutes, "/start": etat_toutes})
    if EXPERT:
        ALERTES.commandes["/scene"] = lambda: cams[actif]["surv"]._scene()

    couleurs_id = {}
    etat = {"fps": 0.0, "ms": 0.0, "mode": 1, "rec": False, "hd": False, "question": "", "reponse": "",
            "vignette": None, "cible": None, "cible_annoncee": False, "surv": cams[actif]["surv"].statut()}
    # ui["mode"] : normal | dessin (entourer un objet) | saisie (taper un nom / une question) | revision | zone
    ui = {"mode": "normal", "saisie": "", "but": None, "dessin": None, "gele": None, "boite": None,
          "dets": [], "det": None, "item": None, "points": []}
    souris = {"down": False, "p0": None, "p1": None}
    voir_trainees, writer = True, None
    frame, dets, compte, compte_p = np.zeros((H, W, 3), np.uint8), [], Counter(), Counter()

    def repondre(txt, img=None, dire=True):
        etat["reponse"], etat["vignette"] = txt, img
        journal.ajouter("IA: " + txt[:34], OR)
        if dire:
            parler(txt)

    def corriger(texte_ou_nom, frame, boite, parts=None, uid=None, autres=()):
        """Applique une correction (niveau 2) : « REJET » = pas un objet, sinon un nom (connu ou nouveau)."""
        if not apprentissage:
            return repondre("L'apprentissage n'est pas disponible.", dire=False)
        if texte_ou_nom == "REJET":
            nom = "REJET"
        elif texte_ou_nom in cerveau.noms_fr:
            nom = texte_ou_nom
        else:
            nom, _ = trouver_objet(_norm(texte_ou_nom), cerveau.noms_fr)
            if nom is None:                                   # nom inconnu → devient un objet perso (en or)
                nom = texte_ou_nom.strip().lower()
                cerveau.apprendre(nom, frame, boite)
        apprentissage.ajouter(nom, frame, boite, parts, autres)
        if uid is not None:
            perception.reverifier(uid)
        n = len(apprentissage.ex)
        if nom == "REJET":
            repondre(f"Compris, ce n'est pas un objet. Je ne le montrerai plus. ({n} corrections)")
        else:
            repondre(f"Merci ! C'est noté : {nom}. ({n} corrections)")

    def autres_surs(sauf_uid):
        """Les autres objets sûrs de l'image, pour que l'image du dataset soit étiquetée en entier."""
        return [(d["nom"], d["b"]) for d in dets if d["uid"] != sauf_uid and not d["incertain"] and not d["perso"]]

    def traiter(txt):
        etat["question"] = txt
        connus = list(dict.fromkeys(cerveau.noms_fr + list(memoire.d)))
        action, obj, libre = comprendre(txt, connus)
        if EXPERT and (action == "inconnu" or
                       (pour_expert(txt) and action not in ("corriger", "oublier", "stop", "mission", "hd", "mode"))):
            return repondre(EXPERT.demander(frame, txt + CONSIGNE), dire=False)   # niveau 4 : Qwen regarde l'image
        if action == "corriger":                            # « non, c'est un X » → l'objet sous le viseur
            cand = [d for d in dets if cerveau.textes_en[cerveau.noms_fr.index(d["nom"])] != "person"] if dets else []
            if not cand:
                return repondre("Je ne vois aucun objet à corriger. Mets-le au centre de l'image.")
            d = min(cand, key=lambda d: ((d["b"][0] + d["b"][2]) / 2 - W / 2) ** 2 + ((d["b"][1] + d["b"][3]) / 2 - H / 2) ** 2)
            if obj is None and not libre:
                return repondre("Dis-moi le bon nom, par exemple : non, c'est un cordon badge.")
            return corriger(obj or libre, frame, d["b"], d["poly"], d["uid"], autres_surs(d["uid"]))
        rep, img = executer(action, obj, libre, {"etat": etat, "memoire": memoire, "mission": mission, "compte": compte})
        repondre(rep, img)

    def revision_suivante():
        if apprentissage and apprentissage.a_revoir:
            ui.update(mode="revision", item=apprentissage.a_revoir.popleft(), saisie="")
        else:
            etait_en_revision = ui["mode"] == "revision"
            ui.update(mode="normal", item=None, saisie="")
            repondre("Révision terminée, merci !" if etait_en_revision else
                     "Rien à revoir pour le moment : je mets de côté les objets dont je ne suis pas sûr.")

    def sur_souris(ev, x, y, flags, _):
        if ui["mode"] == "zone":                         # chaque clic ajoute un coin de la zone
            if ev == cv2.EVENT_LBUTTONDOWN:
                ui["points"].append((min(x, W - 1), y))
            return
        if ui["mode"] != "dessin":
            return
        x = min(x, W - 1)
        if ev == cv2.EVENT_LBUTTONDOWN:
            souris.update(down=True, p0=(x, y), p1=(x, y))
        elif ev == cv2.EVENT_MOUSEMOVE and souris["down"]:
            souris["p1"] = (x, y)
        elif ev == cv2.EVENT_LBUTTONUP and souris["down"]:
            souris["down"] = False
            (x0, y0) = souris["p0"]
            b, det = (min(x0, x), min(y0, y), max(x0, x), max(y0, y)), None
            if b[2] - b[0] < 12 or b[3] - b[1] < 12:     # simple clic → prend l'objet détecté sous la souris
                sous = [d for d in ui["dets"] if d["b"][0] <= x <= d["b"][2] and d["b"][1] <= y <= d["b"][3]]
                det = min(sous, key=lambda d: (d["b"][2] - d["b"][0]) * (d["b"][3] - d["b"][1])) if sous else None
                b = det["b"] if det else None
            if b:
                ui.update(boite=b, det=det, mode="saisie", saisie="",
                          but="nom" if ui["dessin"] == "apprendre" else "correction")

    def bandeau_choix(vue, propositions, prefixe):
        choix = "   ".join(f"{i + 1}) {p}" for i, p in enumerate(propositions[:3]))
        bandeau(vue, f"{prefixe} {choix}   X) pas un objet", OR, H - 110)
        bandeau(vue, f"...ou tape le bon nom : {ui['saisie']}_    (Entree = valider, Echap = annuler)", OR)

    def vignette_camera(vue, n, j, t):
        """Petite image d'une autre caméra, en bas à droite (cadre rouge si une alerte est en cours sur elle)."""
        K = cams[n]
        tw, th = W // 4, H // 4
        x0, y0 = W - tw - 12, H - 40 - (th + 12) * (j + 1)
        v = cv2.resize(K["frame"] if K["frame"] is not None else np.zeros((H, W, 3), np.uint8), (tw, th))
        en_ligne = K["cam"].en_ligne and not K["coupee"]
        if not en_ligne:
            v = (v * 0.35).astype(np.uint8)
        for d in K["dets"]:
            x1, y1, x2, y2 = (int(c / 4) for c in d["b"])
            cv2.rectangle(v, (x1, y1), (x2, y2), couleur(d["nom"], d["perso"]), 1)
        vue[y0:y0 + th, x0:x0 + tw] = v
        alerte = bool(K["surv"].zones_alerte) and int(t * 4) % 2
        cv2.rectangle(vue, (x0 - 2, y0 - 2), (x0 + tw + 1, y0 + th + 1), ROUGE if alerte else CYAN, 2)
        etat_cam = "" if en_ligne else " - COUPEE" if K["coupee"] else " - DECONNECTEE"
        texte(vue, n.upper() + etat_cam, (x0 + 6, y0 + 18), 0.5, CYAN if en_ligne else ROUGE)
        texte(vue, "[Tab]", (x0 + tw - 52, y0 + th - 8), 0.45, BLANC)

    fenetre = "VISION LIVE"
    cv2.namedWindow(fenetre, cv2.WINDOW_NORMAL | cv2.WINDOW_GUI_NORMAL)   # sans la barre d'outils Qt (icônes blanches)
    cv2.resizeWindow(fenetre, W + PW, H)
    cv2.setMouseCallback(fenetre, sur_souris)
    journal.ajouter("Systeme en ligne : " + ", ".join(f"{n}{'' if C['cam'].en_ligne else ' (attente)'}"
                                                       for n, C in cams.items()), VERT)
    if apprentissage and apprentissage.ex:
        journal.ajouter(f"{len(apprentissage.ex)} corrections chargees", OR)
    t_prec = time.perf_counter()

    try:
        while True:
            C = cams[actif]                               # la caméra affichée en grand
            perception, surv, zones, heat, trainees = C["perception"], C["surv"], C["zones"], C["heat"], C["trainees"]
            # la saisie d'un nom de zone ne fige pas l'image : la surveillance continue pendant ce temps
            fige = ui["mode"] in ("dessin", "revision") or (ui["mode"] == "saisie" and ui["but"] not in ("question", "zone"))
            if ui["mode"] == "revision":                  # on montre l'image de l'objet douteux
                frame = cv2.imdecode(ui["item"]["jpg"], cv2.IMREAD_COLOR)
            elif fige:                                    # image figée pendant l'apprentissage / la correction
                frame = ui["gele"].copy()
            else:
                # --- perception sur CHAQUE caméra : suivi + vote + 2e regard + mémoire + règles de surveillance
                compte = Counter()
                for n, K in cams.items():
                    if K["coupee"]:                       # caméra coupée avec D : rien à analyser
                        K["dets"] = []
                        continue
                    f = K["cam"].lire()
                    if f is None:                         # caméra muette : on garde sa dernière image (affichée grisée)
                        K["dets"] = []
                        continue
                    t0 = time.perf_counter()
                    dets_brut = K["perception"].analyser(f, etat["hd"])   # sur l'image d'ORIGINE (iPhone debout...)
                    K["frame"], K["dets"] = adapter(f, W, H), recadrer(dets_brut, f.shape, W, H)
                    f = K["frame"]                        # la suite (mémoire, surveillance, écran) : format fenêtre
                    if n == actif:
                        etat["ms"] = 0.9 * etat["ms"] + 0.1 * (time.perf_counter() - t0) * 1000
                    surs = [d for d in K["dets"] if not d["incertain"]]   # seuls les objets sûrs vont en mémoire / journal
                    compte.update(d["nom"] for d in surs)
                    K["heat"].maj([d["b"] for d in K["dets"]])
                    memoire.maj([(d["nom"], d["b"]) for d in surs], f, n if plusieurs else None)
                    for msg in K["surv"].maj(f, K["dets"]):              # règles de surveillance → alertes
                        repondre(msg)
                        journal.ajouter(msg[:44], ROUGE)
                    actifs = set()
                    for d in K["dets"]:
                        if not d["incertain"]:
                            journal.observer(d["uid"], d["nom"], d["conf"], d["age"], d["perso"])
                        actifs.add(d["uid"])
                        couleurs_id[d["uid"]] = couleur(d["nom"], d["perso"])
                        x1, y1, x2, y2 = d["b"]
                        K["trainees"][d["uid"]].append(((x1 + x2) // 2, (y1 + y2) // 2))
                    for u in list(K["trainees"]):
                        if u not in actifs:
                            K["trainees"][u].popleft() if K["trainees"][u] else K["trainees"].pop(u)
                frame = C["frame"] if C["frame"] is not None else np.zeros((H, W, 3), np.uint8)
                dets = C["dets"]
                compte_p = Counter((d["etiquette"], d["perso"]) for d in dets)
                msg = mission.maj(set(compte))
                if msg:
                    repondre(msg)
                if etat["cible"] in compte and not etat["cible_annoncee"]:
                    etat["cible_annoncee"] = True
                    repondre(f"{etat['cible']} trouvé !")
            etat["surv"] = surv.statut()

            # --- rendu
            m = etat["mode"]
            if ui["mode"] == "revision":
                vue = frame.copy()
            elif m == 2: vue = mode_spotlight(frame, [d["poly"] for d in dets], [d["b"] for d in dets])
            elif m == 3: vue = heat.rendu(frame)
            elif m == 4: vue = mode_nocturne(frame)
            else:        vue = frame.copy()
            if ui["mode"] != "revision":
                zones.dessiner(vue, surv.zones_alerte)
            if voir_trainees and not fige:
                dessiner_trainees(vue, trainees, couleurs_id)
            t = time.time()
            for d in ([] if ui["mode"] == "revision" else dets):
                x1, y1, x2, y2 = d["b"]
                c = VERT if m == 4 and not d["perso"] else couleur(d["nom"], d["perso"])
                if d["incertain"]:
                    c = tuple(int(0.45 * v + 0.55 * 160) for v in c)   # couleur éteinte = « je ne suis pas sûr »
                cible = d["nom"] in (etat["cible"], mission.cible)
                if m in (1, 4):
                    contour(vue, d["poly"], c, 0.10 if d["incertain"] else 0.18)
                cadre_coins(vue, x1, y1, x2, y2, c, 1 if d["incertain"] else 3 if cible or d["perso"] else 2)
                tag = ("* " if d["perso"] else "") + d["etiquette"] + (" (appris)" if d.get("souvenir") else "")
                etiquette(vue, x1, y1, tag if d["incertain"] else f"{tag} {d['conf'] * 100:.0f}%", c)
                if d["nom"] == etat["cible"]:
                    verrouillage(vue, x1, y1, x2, y2, t)

            if ui["mode"] != "revision" and C["coupee"] and not fige:                 # caméra active coupée (D)
                vue = (vue * 0.4).astype(np.uint8)
                bandeau(vue, f"CAMERA {actif.upper()} COUPEE - [D] pour la rallumer", ROUGE, H // 2)
            elif ui["mode"] != "revision" and not C["cam"].en_ligne and not fige:   # caméra active déconnectée
                vue = (vue * 0.4).astype(np.uint8)
                bandeau(vue, f"CAMERA {actif.upper()} DECONNECTEE - reconnexion automatique...", ROUGE, H // 2)
            viseur_central(vue, t)
            texte(vue, datetime.now().strftime("%d/%m/%Y  %H:%M:%S"), (14, 28), 0.6, CYAN)
            if plusieurs and ui["mode"] != "revision":
                texte(vue, f"CAMERA {actif.upper()}   [Tab] changer   [D] couper/rallumer", (14, 54), 0.55, CYAN)
                for j, n in enumerate(x for x in noms if x != actif):
                    vignette_camera(vue, n, j, t)
            texte(vue, f"MODE {m} : {MODES[m]}" + ("   |   HAUTE PRECISION" if etat["hd"] else ""), (14, H - 16), 0.6, CYAN)
            s_txt, s_c = etat["surv"]
            if s_c != GRIS:                               # surveillance armée (ou en cours d'armement)
                texte(vue, "SURVEILLANCE " + s_txt.split("|")[0].strip(), (W // 2 - 170, 28), 0.65, s_c, 2)
                if s_c == ROUGE and surv.zones_alerte and int(t * 4) % 2:
                    cv2.rectangle(vue, (0, 0), (W - 1, H - 1), ROUGE, 8)
            if EXPERT and EXPERT.etat in ("chargement", "reflexion"):
                points = "." * (1 + int(t * 2) % 3)
                texte(vue, ("QWEN SE REVEILLE" if EXPERT.etat == "chargement" else "QWEN REFLECHIT") + points,
                      (W - 290, 86), 0.65, OR, 2)
            if etat["cible"] and etat["cible"] not in compte and not fige:
                yb = int((t * 300) % H)                 # ligne de balayage "radar"
                cv2.line(vue, (0, yb), (W, yb), ROUGE, 1)
                bandeau(vue, f"RECHERCHE : {etat['cible']} ...   (dis 'stop' ou ecris 'stop' pour arreter)", ROUGE, 60)
            if time.time() - mission.flash < 0.6:
                cv2.rectangle(vue, (0, 0), (W - 1, H - 1), VERT, 14)
                texte(vue, "MISSION REUSSIE !", (W // 2 - 190, H // 2 - 60), 1.4, VERT, 3)
            if ui["mode"] == "dessin":
                quoi = "APPRENTISSAGE" if ui["dessin"] == "apprendre" else "CORRECTION"
                bandeau(vue, f"{quoi} : clique sur l'objet (ou entoure-le a la souris)  -  Echap = annuler", OR)
                if souris["down"]:
                    cv2.rectangle(vue, souris["p0"], souris["p1"], OR, 2)
            elif ui["mode"] == "zone" or (ui["mode"] == "saisie" and ui["but"] == "zone"):
                pts = ui["points"]
                for p in pts:
                    cv2.circle(vue, p, 6, ORANGE, -1)
                if len(pts) > 1:
                    cv2.polylines(vue, [np.array(pts, np.int32)], len(pts) > 2, ORANGE, 2, cv2.LINE_AA)
                if ui["mode"] == "zone":
                    bandeau(vue, f"ZONE : clique les coins ({len(pts)})  Entree = finir  Retour = effacer  Echap = annuler", OR)
                else:
                    bandeau(vue, f"Nom de la zone (ex. bureau, porte) : {ui['saisie']}_     (Entree = valider)", OR)
            elif ui["mode"] == "saisie":
                if ui["but"] == "nom":
                    cv2.rectangle(vue, ui["boite"][:2], ui["boite"][2:], OR, 2)
                    bandeau(vue, f"Nom de cet objet : {ui['saisie']}_     (Entree = valider, Echap = annuler)", OR)
                elif ui["but"] == "correction":
                    cv2.rectangle(vue, ui["boite"][:2], ui["boite"][2:], OR, 3)
                    props = (ui["det"] or {}).get("propositions") or []
                    bandeau_choix(vue, props, "C'est quoi ?")
                else:
                    bandeau(vue, f"Ta question : {ui['saisie']}_     (Entree = envoyer)", CYAN)
            elif ui["mode"] == "revision":
                it = ui["item"]
                x1, y1, x2, y2 = it["boite"]
                cv2.rectangle(vue, (x1, y1), (x2, y2), OR, 3)
                texte(vue, f"REVISION  ({len(apprentissage.a_revoir)} restants)  -  Espace = passer", (14, 60), 0.7, OR, 2)
                bandeau_choix(vue, it["propositions"], "Je ne suis pas sur. C'est :")
            if voix.etat == "ecoute":
                cv2.circle(vue, (W - 40, 40), 14 + int(4 * np.sin(t * 10)), ROUGE, -1)
                texte(vue, "J'ECOUTE", (W - 150, 46), 0.6, ROUGE)

            # --- assemblage + affichage
            tp = time.perf_counter()
            etat["fps"] = 0.9 * etat["fps"] + 0.1 / max(tp - t_prec, 1e-6)
            t_prec = tp
            canvas[:, :W] = vue
            dessiner_panneau(canvas[:, W:], etat, compte_p, journal, mission, voix)
            cv2.imshow(fenetre, canvas)
            if writer is not None:
                writer.write(canvas)

            # --- voix
            voix.maj()
            while not voix.resultats.empty():
                txt = voix.resultats.get()
                if not txt:
                    repondre("Je n'ai rien entendu. Appuie sur V et parle plus près du micro.")
                elif txt.startswith("__erreur__"):
                    repondre("Erreur micro : " + txt[11:], dire=False)
                else:
                    traiter(txt)
            while EXPERT and not EXPERT.resultats.empty():     # réponses de Qwen (niveau 4)
                repondre(EXPERT.resultats.get())

            # --- clavier
            k = cv2.waitKey(1) & 0xFF
            try:
                if cv2.getWindowProperty(fenetre, cv2.WND_PROP_VISIBLE) < 1:
                    break
            except cv2.error:                             # fenêtre fermée avec la croix
                break
            choix_mode = ui["mode"] == "revision" or (ui["mode"] == "saisie" and ui["but"] == "correction")
            if choix_mode:
                if ui["mode"] == "revision":
                    it = ui["item"]
                    img, boite, parts, props, uid, autres = (cv2.imdecode(it["jpg"], cv2.IMREAD_COLOR), it["boite"],
                                                             it["parts"], it["propositions"], None, ())
                else:
                    det = ui["det"] or {}
                    img, boite, parts, props, uid = ui["gele"], ui["boite"], det.get("poly"), det.get("propositions") or [], det.get("uid")
                    autres = autres_surs(uid)
                choix = None
                if not ui["saisie"] and ord("1") <= k <= ord("3") and k - ord("1") < len(props):
                    choix = props[k - ord("1")]
                elif not ui["saisie"] and k in (ord("x"), ord("X")):
                    choix = "REJET"
                elif k in (13, 10) and ui["saisie"].strip():
                    choix = ui["saisie"].strip()
                elif k == 32 and not ui["saisie"] and ui["mode"] == "revision":
                    revision_suivante(); continue
                elif k == 27:
                    ui.update(mode="normal", saisie="", item=None); continue
                elif k == 8:
                    ui["saisie"] = ui["saisie"][:-1]
                elif 32 <= k < 127:
                    ui["saisie"] += chr(k)
                if choix:
                    corriger(choix, img, boite, parts, uid, autres)
                    if ui["mode"] == "revision":
                        revision_suivante()
                    else:
                        ui.update(mode="normal", saisie="")
                continue
            if ui["mode"] == "saisie":
                if k in (13, 10):
                    s = ui["saisie"].strip()
                    if ui["but"] == "nom" and s:
                        nb = cerveau.apprendre(s.lower(), ui["gele"], ui["boite"])
                        repondre(f"C'est noté ! Je reconnais maintenant : {s}. {nb} exemple{'s' if nb > 1 else ''}. "
                                 "Appuie encore sur A pour me le montrer sous un autre angle.")
                    elif ui["but"] == "question" and s:
                        traiter(s)
                    elif ui["but"] == "zone" and s:
                        zones.ajouter(s.lower(), ui["points"], W, H)
                        sur = f" sur la caméra {actif}" if plusieurs else ""
                        repondre(f"Zone {s} enregistrée{sur}. La surveillance ne regarde plus que les zones dessinées.")
                    ui.update(mode="normal", saisie="", points=[])
                elif k == 27:
                    ui.update(mode="normal", saisie="", points=[])
                elif k == 8:
                    ui["saisie"] = ui["saisie"][:-1]
                elif 32 <= k < 127:
                    ui["saisie"] += chr(k)
                continue
            if ui["mode"] == "zone":
                if k in (13, 10) and len(ui["points"]) >= 3:
                    ui.update(mode="saisie", but="zone", saisie="")
                elif k == 8:                              # Retour : efface le dernier coin, ou la dernière zone
                    if ui["points"]:
                        ui["points"].pop()
                    else:
                        z = zones.retirer_derniere()
                        repondre(f"Zone {z} supprimée." if z else "Aucune zone à supprimer.", dire=False)
                elif k == 27:
                    ui.update(mode="normal", points=[])
                continue
            if ui["mode"] == "dessin":
                if k == 27:
                    ui["mode"] = "normal"
                continue
            if k in (ord("q"), 27):
                break
            elif k == 9 and plusieurs:                    # Tab : caméra suivante en grand
                actif = noms[(noms.index(actif) + 1) % len(noms)]
                repondre(f"Caméra {actif}.", dire=False)
            elif k == ord("d"):                           # couper / rallumer la caméra affichée en grand
                K = cams[actif]
                if K["coupee"]:
                    K["cam"], K["coupee"] = Camera(actif, CONFIG["cameras"][actif]), False
                    repondre(f"Caméra {actif} rallumée.", dire=False)
                else:
                    K["coupee"], K["dets"] = True, []
                    K["trainees"].clear()
                    threading.Thread(target=K["cam"].stop, daemon=True).start()   # libère la webcam / le flux sans figer
                    repondre(f"Caméra {actif} coupée. D pour la rallumer" + (", Tab pour voir l'autre." if plusieurs else "."),
                             dire=False)
            elif k in (ord("1"), ord("2"), ord("3"), ord("4")):
                etat["mode"] = k - ord("0")
            elif k == ord("v"):
                if voix.etat == "ecoute":
                    voix.arreter()
                elif not voix.demarrer():
                    journal.ajouter("Voix pas encore prete" if voix.etat == "chargement" else f"Voix: {voix.erreur}", GRIS)
            elif k == ord("f"):
                ui.update(mode="saisie", but="question", saisie="")
            elif k == ord("i"):                           # niveau 4 : « que se passe-t-il ? » sur l'image en direct
                etat["question"] = "Que se passe-t-il ?"
                repondre(EXPERT.demander(frame) if EXPERT else "Qwen est désactivé (CONFIG['expert']).", dire=False)
            elif k in (ord("a"), ord("c")):
                ui.update(mode="dessin", dessin="apprendre" if k == ord("a") else "corriger",
                          gele=frame.copy(), dets=list(dets))
                souris["down"] = False
            elif k == ord("g"):                           # toutes les caméras ensemble
                repondre(basculer_tout())
            elif k == ord("z"):
                ui.update(mode="zone", points=[])
                repondre("Clique les coins de la zone, puis Entrée. Retour efface.", dire=False)
            elif k == ord("e"):
                revision_suivante()
            elif k == ord("u"):
                if apprentissage:
                    repondre(apprentissage.entrainer().capitalize() + ".")
            elif k == ord("m"):
                repondre(f"Mission : montre-moi {mission.nouvelle()} !")
            elif k == ord("h"):
                etat["hd"] = not etat["hd"]
            elif k == ord("t"):
                voir_trainees = not voir_trainees
            elif k in (ord("+"), ord("=")):
                CONFIG["conf"] = min(0.95, round(CONFIG["conf"] + 0.05, 2))
            elif k == ord("-"):
                CONFIG["conf"] = max(0.05, round(CONFIG["conf"] - 0.05, 2))
            elif k == ord("s"):
                cv2.imwrite(str(CAPTURES / f"capture_{datetime.now():%Y%m%d_%H%M%S}.png"), canvas)
                journal.ajouter("Capture enregistree", CYAN)
            elif k == ord("r"):                           # vidéo de l'écran en H.264 (cellule 6b) : lisible partout
                if writer is None:
                    writer = VideoH264(CAPTURES / f"video_{datetime.now():%Y%m%d_%H%M%S}.mp4", W + PW, H)
                    journal.ajouter("Enregistrement...", ROUGE)
                else:
                    writer.release(); writer = None
                    journal.ajouter("Video enregistree", CYAN)
                etat["rec"] = writer is not None
    finally:
        if voix.proc:
            voix.arreter()
        for K in cams.values():
            K["surv"].arreter()                           # termine le clip en cours avant de fermer
            K["cam"].stop()
        if writer is not None:
            writer.release()
        journal.fermer()
        memoire.sauver()
        cv2.destroyAllWindows()
        for _ in range(5):
            cv2.waitKey(1)

    rejets = sum(K["perception"].rejets for K in cams.values())
    print(f"Session terminée — {len(journal.vus)} objets suivis | {rejets} fausses détections écartées"
          f" par le vérificateur | score mission : {mission.score}")
    if apprentissage:
        print(f"Apprentissage : {len(apprentissage.ex)} corrections au total | {apprentissage.bilan}")
    print("Top objets :", ", ".join(f"{n}×{k}" for n, k in journal.total.most_common(10)) or "-")
    print("Objets appris :", ", ".join(f"{n} ({cerveau.perso_n[n]} ex.)" for n in cerveau.perso) or "aucun")
    print("Mémoire :", len(memoire.d), "objets localisés →", Memoire.FICHIER)
    for n, K in cams.items():
        print(f"Caméra {n} : {len(K['surv'].evenements)} alertes | {K['cam'].reconnexions} reconnexions")
    print(f"Alertes en attente d'envoi : {len(ALERTES.en_attente)} →", EVENEMENTS)
    if EXPERT:
        print(f"Qwen : {EXPERT.n} réponses" + (f" | erreur : {EXPERT.erreur}" if EXPERT.erreur else ""))

In [ ]:
# 11) ▶️ LANCER — une fenêtre s'ouvre. Clique dessus puis utilise le clavier (Q pour quitter)
lancer()